# Enveda CASMI 2026 — Notebook 01
## Data Understanding, Scientific EDA, Validation Design & Modeling Foundation

**Task.** Given one or more LC-MS/MS spectra of an unknown small molecule, return up to 25 candidate structures (SMILES), ranked. **Metric:** MRR@25, where a candidate is correct if its *connectivity* matches the label.

**This notebook trains no model.** It establishes the rules of the project: what an observation is, how identity is defined, what the physics of precursor mass allows, how ambiguous the candidate space is, how train differs from test, and which folds every later experiment must use.

**Section protocol.** Every major section follows `QUESTION → ANALYSIS → RESULT → WHY IT MATTERS → DECISION / ARTIFACT`. The QUESTION and WHY IT MATTERS are written in markdown. The RESULT and DECISION are *printed by code from measured values* (through `report()`), so the narrative cannot drift away from the data when the data or configuration change.

**Run protocol.** `Restart Kernel → Run All`, or headless:
`jupyter nbconvert --to notebook --execute --inplace 01_data_understanding.ipynb`
The final checklist (§42) verifies that the kernel was fresh and every code cell ran exactly once, in order.

## 0. Reproducibility first

In [ ]:
import sys, os, re, json, math, time, hashlib, random, platform, warnings, datetime, contextlib, gzip
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
from scipy import stats
from scipy.spatial.distance import jensenshannon
import sklearn
from sklearn.metrics import roc_auc_score
import matplotlib
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.dataset as pads
import pyarrow.compute as pc

import rdkit
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import rdFingerprintGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

# Clean-run bookkeeping (verified in §42). The IPython input history `In` is used rather than
# execution_count, whose off-by-one convention differs between IPython/ipykernel versions.
# Fresh kernel <=> no input was executed before this cell.
try:
    _IP = get_ipython()  # noqa: F821
    _HIST_AT_START = len(_IP.user_ns["In"])            # includes this cell
    _FRESH_KERNEL = not any(str(x).strip() for x in _IP.user_ns["In"][:-1])
except (NameError, KeyError):
    _IP, _HIST_AT_START, _FRESH_KERNEL = None, None, False

VERSIONS = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "Matplotlib": matplotlib.__version__,
    "RDKit": rdkit.__version__,
    "PyArrow": pa.__version__,
}
OPTIONAL = {}
for _mod in ("joblib", "matchms", "polars", "duckdb"):
    try:
        OPTIONAL[_mod] = __import__(_mod).__version__
    except Exception:
        OPTIONAL[_mod] = None
VERSIONS.update({k: v for k, v in OPTIONAL.items() if v})

print(f"SEED = {SEED}")
for k, v in VERSIONS.items():
    print(f"  {k:<14} {v}")
print("  not installed:", [k for k, v in OPTIONAL.items() if v is None])

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)


@contextlib.contextmanager
def rdkit_quiet():
    """Silence RDKit parse messages *locally*; invalid SMILES are still counted explicitly."""
    RDLogger.DisableLog("rdApp.*")
    try:
        yield
    finally:
        RDLogger.EnableLog("rdApp.*")

## 1. Local project configuration

Only `INPUT_ROOT` needs to change to run elsewhere (an environment variable `CASMI_INPUT_ROOT` may override it for headless runs). All analysis knobs live in `CONFIG`; `CONFIG` is hashed into every cache's provenance so a changed knob invalidates the cache.

In [ ]:
def _find_repo_root(marker="data"):
    """Walk upward from the current working directory until a folder containing
    `marker` is found. Jupyter's CWD is normally wherever the notebook file lives, so
    a bare relative path ("data") only resolves correctly when the notebook happens to
    be launched from the repo root -- this makes it work from notebooks/ (or deeper) too.
    """
    d = Path.cwd().resolve()
    for _ in range(4):
        if (d / marker).exists():
            return d
        d = d.parent
    return Path.cwd()


_REPO_ROOT = _find_repo_root()
INPUT_ROOT = _REPO_ROOT / "data"
OUTPUT_ROOT = _REPO_ROOT / "outputs"
CACHE_ROOT = OUTPUT_ROOT / "cache"
FIGURE_ROOT = OUTPUT_ROOT / "figures"

if os.environ.get("CASMI_INPUT_ROOT"):
    INPUT_ROOT = Path(os.environ["CASMI_INPUT_ROOT"])
for _d in (OUTPUT_ROOT, CACHE_ROOT, FIGURE_ROOT):
    _d.mkdir(parents=True, exist_ok=True)
assert INPUT_ROOT.exists(), f"INPUT_ROOT does not exist: {INPUT_ROOT.resolve()}"

CONFIG = dict(
    n_folds=5,
    n_jobs=max(1, min(8, (os.cpu_count() or 2) - 1)),
    tautomer_max_tautomers=None,        # None = RDKit default (closest to the competition's canonicalizer)
    # bounded peak-array sample (§18)
    peak_sample_train_spectra=120_000,
    peak_sample_train_molecules=6_000,
    peak_sample_spectra_per_molecule=6,
    peak_sample_test_max=60_000,
    # spectral audit (§17)
    fragment_above_precursor_da=5.0,
    extreme_fragment_mz=5000.0,
    # similarity study (§21-24)
    pair_mass_tol_ppm=10.0,
    n_pair_anchors=6_000,
    n_pairs_peakaware=2_500,
    peak_match_tol_da=0.02,
    bin_widths=[0.05, 0.1, 0.2, 0.5, 1.0],
    provisional_bin_width=0.1,
    max_peaks_similarity=200,
    # structure landscape (§27)
    nn_query_sample=5_000,
    # mass physics (§15, §29)
    tolerance_grid_ppm=[1, 2, 3, 5, 10, 20, 50],
    ambiguity_tols_ppm=[1, 2, 5, 10, 20],
    target_capture=0.99,
    rare_adduct_frac=0.001,
    # candidate lists (§35)
    max_candidates=25,
)
COLMAP_OVERRIDES = {"train": {}, "test": {}}   # e.g. {"train": {"adduct_col": "precursor_type"}}
FILE_OVERRIDES = {}                              # e.g. {"train": INPUT_ROOT / "train.parquet"}

RESULTS, CHECKS, FIGURES, ARTIFACTS = {}, {}, [], {}


def report(section, result, decision=None):
    """Print a RESULT block (and optional DECISION) and store it in RESULTS[section]."""
    print(f"── RESULT [{section}] " + "─" * max(4, 60 - len(section)))
    if isinstance(result, dict):
        for k, v in result.items():
            if isinstance(v, float):
                v = f"{v:.4g}"
            print(f"  {k}: {v}")
        RESULTS.setdefault(section, {}).update(result)
    else:
        print("  " + str(result).replace("\n", "\n  "))
        RESULTS.setdefault(section, {})["text"] = str(result)
    if decision:
        print(f"── DECISION / ARTIFACT [{section}]")
        print("  " + decision.replace("\n", "\n  "))
        RESULTS[section]["decision"] = decision


def savefig(fig, name):
    path = FIGURE_ROOT / f"{name}.png"
    fig.savefig(path, dpi=110, bbox_inches="tight")
    FIGURES.append(str(path))
    plt.show()
    plt.close(fig)


def qsummary(x, qs=(0, 0.5, 0.9, 0.95, 0.99, 1.0), names=None):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    names = names or [("min" if q == 0 else "max" if q == 1 else f"p{q*100:g}") for q in qs]
    if len(x) == 0:
        return {n: np.nan for n in names}
    return dict(zip(names, np.quantile(x, qs)))


def check(name, ok, detail=""):
    CHECKS[name] = bool(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""))
    return bool(ok)

### Provenance utilities (used from §5 onward, formalised in §38)

Every cache is written with a sidecar `*.meta.json` containing a timestamp, input-file fingerprints (size + SHA-256 of head/tail bytes), SEED, library versions, the configuration hash and row count. `artifact_is_fresh()` refuses a cache whose inputs or configuration changed, so stale caches cannot silently survive.

In [ ]:
PIPELINE_VERSION = "nb01-v1"   # bump when preprocessing logic changes


def file_fingerprint(path, chunk=1 << 20):
    path = Path(path)
    if path.is_dir():
        files = sorted(p for p in path.rglob("*") if p.is_file())
        h = hashlib.sha256()
        for f in files:
            h.update(json.dumps(file_fingerprint(f), sort_keys=True).encode())
        return {"path": str(path), "kind": "dir", "n_files": len(files), "sha256": h.hexdigest()}
    st = path.stat()
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        h.update(fh.read(chunk))
        if st.st_size > 2 * chunk:
            fh.seek(-chunk, 2)
            h.update(fh.read(chunk))
    return {"path": str(path), "size": st.st_size, "mtime": int(st.st_mtime), "sha256": h.hexdigest()}


def _fp_identity(fp):
    return {k: v for k, v in fp.items() if k != "mtime"}


def config_hash(cfg):
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]


INPUT_FINGERPRINTS = {}   # filled in §3


def _meta_for(name, n_rows, config, extra):
    return {
        "artifact": name,
        "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "pipeline_version": PIPELINE_VERSION,
        "input_fingerprints": INPUT_FINGERPRINTS,
        "SEED": SEED,
        "versions": VERSIONS,
        "config": config if config is not None else CONFIG,
        "config_hash": config_hash(config if config is not None else CONFIG),
        "row_count": int(n_rows),
        **(extra or {}),
    }


def _make_parquet_safe(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            sample = out[c].dropna().head(200)
            if len(sample) and not all(isinstance(v, str) for v in sample):
                out[c] = out[c].map(lambda v: None if v is None else (v if isinstance(v, str) else str(v)))
    return out


def save_artifact(df, name, config=None, extra=None):
    path = CACHE_ROOT / f"{name}.parquet"
    _make_parquet_safe(df).to_parquet(path, index=False)
    meta = _meta_for(name, len(df), config, {**(extra or {}), "columns": list(map(str, df.columns))})
    (CACHE_ROOT / f"{name}.meta.json").write_text(json.dumps(meta, indent=2, default=str))
    ARTIFACTS[name] = str(path)
    return path


def save_array_artifact(arrays: dict, name, config=None, extra=None):
    path = CACHE_ROOT / f"{name}.npz"
    np.savez_compressed(path, **arrays)
    n = len(next(iter(arrays.values()))) if arrays else 0
    meta = _meta_for(name, n, config, {**(extra or {}), "arrays": {k: [str(v.dtype), list(v.shape)] for k, v in arrays.items()}})
    (CACHE_ROOT / f"{name}.meta.json").write_text(json.dumps(meta, indent=2, default=str))
    ARTIFACTS[name] = str(path)
    return path


def artifact_is_fresh(name, config=None, suffix=".parquet"):
    path, mpath = CACHE_ROOT / f"{name}{suffix}", CACHE_ROOT / f"{name}.meta.json"
    if not (path.exists() and mpath.exists()):
        return False
    meta = json.loads(mpath.read_text())
    if meta.get("pipeline_version") != PIPELINE_VERSION:
        return False
    if meta.get("config_hash") != config_hash(config if config is not None else CONFIG):
        return False
    old = {k: _fp_identity(v) for k, v in meta.get("input_fingerprints", {}).items()}
    new = {k: _fp_identity(v) for k, v in INPUT_FINGERPRINTS.items()}
    return old == new

## 2. Discover the actual dataset

**QUESTION.** Which files exist, and which of them hold training metadata, test metadata, peak arrays, labels and the sample submission?

**WHY IT MATTERS.** Nothing downstream may assume filenames or schemas. The inventory is kept compact: its only purpose is to decide how to load.

In [ ]:
COMPRESS_EXT = {".gz", ".bz2", ".xz", ".zip"}
TABULAR_EXT = {".parquet", ".pq", ".csv", ".tsv", ".feather", ".json", ".jsonl", ".pkl", ".pickle", ".mgf"}


def ext_of(p) -> str:
    sfx = [s.lower() for s in Path(p).suffixes]
    if sfx and sfx[-1] in COMPRESS_EXT:
        sfx = sfx[:-1]
    return sfx[-1] if sfx else ""


def _is_parquet_path(p):
    p = Path(p)
    return (p.is_dir() and any(q.suffix.lower() in (".parquet", ".pq") for q in p.rglob("*"))) or ext_of(p) in (".parquet", ".pq")


def parse_mgf(path, max_spectra=None):
    """Minimal MGF reader: one dict per spectrum with header keys (lower-case) + 'mz'/'intensity' arrays."""
    opener = gzip.open if str(path).endswith(".gz") else open
    recs, cur, mzs, ints = [], None, [], []
    with opener(path, "rt", errors="replace") as fh:
        for line in fh:
            line = line.strip()
            if line == "BEGIN IONS":
                cur, mzs, ints = {}, [], []
            elif line == "END IONS":
                cur["mz"], cur["intensity"] = np.asarray(mzs, float), np.asarray(ints, float)
                recs.append(cur)
                cur = None
                if max_spectra and len(recs) >= max_spectra:
                    break
            elif cur is not None and line:
                if "=" in line and not line[0].isdigit():
                    k, v = line.split("=", 1)
                    cur[k.strip().lower()] = v.strip()
                else:
                    parts = re.split(r"[\s,;]+", line)
                    if len(parts) >= 2:
                        mzs.append(float(parts[0])); ints.append(float(parts[1]))
    return pd.DataFrame(recs)


def _csv_kwargs(path):
    return {"sep": "\t"} if ext_of(path) == ".tsv" else {}


def load_table(path, columns=None, nrows=None):
    path = Path(path)
    e = ".parquet" if path.is_dir() else ext_of(path)
    if e in (".parquet", ".pq"):
        dset = pads.dataset(str(path), format="parquet")
        tbl = dset.head(nrows, columns=columns) if nrows else dset.to_table(columns=columns)
        return tbl.to_pandas()
    if e in (".csv", ".tsv"):
        return pd.read_csv(path, usecols=columns, nrows=nrows, **_csv_kwargs(path))
    if e == ".feather":
        df = pd.read_feather(path, columns=columns)
    elif e in (".json", ".jsonl"):
        df = pd.read_json(path, lines=(e == ".jsonl"), nrows=nrows if e == ".jsonl" else None)
    elif e in (".pkl", ".pickle"):
        df = pd.read_pickle(path)
    elif e == ".mgf":
        df = parse_mgf(path, max_spectra=nrows)
    else:
        raise ValueError(f"Unsupported file type: {path}")
    if columns is not None:
        df = df[[c for c in columns if c in df.columns]]
    return df.head(nrows) if nrows else df


def get_columns(path):
    path = Path(path)
    if _is_parquet_path(path):
        return list(pads.dataset(str(path), format="parquet").schema.names)
    if ext_of(path) in (".csv", ".tsv"):
        return list(pd.read_csv(path, nrows=0, **_csv_kwargs(path)).columns)
    return list(load_table(path, nrows=50).columns)


def count_rows(path):
    path = Path(path)
    try:
        if _is_parquet_path(path):
            return pads.dataset(str(path), format="parquet").count_rows()
        if ext_of(path) in (".csv", ".tsv") and path.stat().st_size < 500e6 and not str(path).endswith(".gz"):
            with open(path, "rb") as fh:
                return max(0, sum(1 for _ in fh) - 1)
        if ext_of(path) == ".mgf" and path.stat().st_size < 500e6:
            opener = gzip.open if str(path).endswith(".gz") else open
            with opener(path, "rt", errors="replace") as fh:
                return sum(1 for l in fh if l.startswith("BEGIN IONS"))
    except Exception:
        return None
    return None


_inv = []
for f in sorted(INPUT_ROOT.rglob("*")):
    if not f.is_file() or f.name.startswith("."):
        continue
    e = ext_of(f)
    row = {"rel": str(f.relative_to(INPUT_ROOT)), "path": str(f), "ext": e,
           "size_mb": round(f.stat().st_size / 1e6, 2), "tabular": e in TABULAR_EXT,
           "n_rows": None, "n_cols": None, "columns": ""}
    if row["tabular"]:
        try:
            cols = get_columns(f)
            row.update(n_rows=count_rows(f), n_cols=len(cols), columns=", ".join(map(str, cols[:14])) + (" …" if len(cols) > 14 else ""))
        except Exception as ex:
            row["columns"] = f"<unreadable: {type(ex).__name__}>"
    _inv.append(row)
INVENTORY = pd.DataFrame(_inv)
assert len(INVENTORY), f"No files found under {INPUT_ROOT}"
display(INVENTORY.drop(columns=["path"]))  # noqa: F821

## 3. Build the dataset map

**QUESTION.** Which physical columns play which scientific role?

**ANALYSIS.** Column roles are matched against lists of known names (exact match after normalising case and punctuation — no fuzzy substring matching, which would e.g. confuse `precursor_mz` with an m/z array). Array-valued columns are detected from the *values*, not the names. Unavailable roles are `None`; nothing is invented. `COLMAP_OVERRIDES` / `FILE_OVERRIDES` in §1 correct any wrong guess without touching the analysis code.

In [ ]:
def _pick_file(keyword, exclude=()):
    if keyword in FILE_OVERRIDES:
        return Path(FILE_OVERRIDES[keyword])
    c = INVENTORY[INVENTORY.tabular]
    low = c.rel.str.lower()
    mask = low.str.contains(keyword, regex=False)
    for e in exclude:
        mask &= ~low.str.contains(e, regex=False)
    c = c[mask]
    if c.empty:
        return None
    parents = {Path(p).parent for p in c.path}
    if len(c) > 1 and len(parents) == 1 and set(c.ext) <= {".parquet", ".pq"}:
        return next(iter(parents))            # sharded parquet directory
    return Path(c.sort_values("size_mb").iloc[-1].path)


FILES = {
    "train": _pick_file("train", exclude=("sample", "submission")),
    "test": _pick_file("test", exclude=("sample", "submission")),
    "sample": _pick_file("sample_submission") or _pick_file("submission"),
}
_used = {str(p) for p in FILES.values() if p is not None}
_lib = None
for kw in ("candidate", "library", "structures", "pubchem", "coconut"):
    _lib = _pick_file(kw, exclude=("train", "test", "submission"))
    if _lib is not None and str(_lib) not in _used:
        break
    _lib = None
FILES["candidate_library"] = _lib

ROLE_CANDIDATES = {
    "spectrum_id_col": ["spectrum_id", "spec_id", "spectrumid", "scan_id", "spectrum", "identifier", "id"],
    "molecule_id_col": ["molecule_id", "mol_id", "query_id", "compound_id", "group_id", "challenge_id", "molecule"],
    "smiles_col": ["smiles", "canonical_smiles", "isomeric_smiles", "target_smiles", "label_smiles"],
    "precursor_mz_col": ["precursor_mz", "precursormz", "precursor_m/z", "pepmass", "prec_mz", "precursor_mass", "mz_precursor"],
    "mz_array_col": ["mzs", "mz_array", "mz_values", "peaks_mz", "mz_list", "masses", "mz"],
    "intensity_array_col": ["intensities", "intensity_array", "intensity_values", "peaks_intensity", "intensity_list", "intensity", "ints"],
    "peaks_col": ["peaks", "peak_list", "spectrum_peaks"],
    "adduct_col": ["adduct", "precursor_type", "precursortype", "ion_type", "adducts", "precursor_adduct"],
    "ion_mode_col": ["ion_mode", "ionmode", "polarity", "ionization_mode", "ion_polarity", "mode"],
    "collision_energy_col": ["collision_energy", "collisionenergy", "ce", "energy", "nce", "collision_energy_ev"],
    "instrument_col": ["instrument", "instrument_type", "instrumenttype", "ms_instrument", "mass_analyzer"],
    "source_col": ["source", "data_source", "dataset", "library", "origin", "database", "source_library"],
    "ppm_error_col": ["ppm_error", "precursor_ppm_error", "mass_error_ppm", "delta_ppm", "ppm"],
    "charge_col": ["charge", "precursor_charge"],
    "formula_col": ["formula", "molecular_formula"],
    "inchikey_col": ["inchikey", "inchi_key", "inchikey14", "inchikey_14"],
}
ARRAY_ROLES = ("mz_array_col", "intensity_array_col", "peaks_col")


def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def guess_col(columns, role):
    nm = {}
    for c in columns:
        nm.setdefault(_norm(c), c)
    for cand in ROLE_CANDIDATES[role]:
        if _norm(cand) in nm:
            return nm[_norm(cand)]
    return None


def looks_array(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return True
    if isinstance(v, str):
        s = v.strip()
        return len(s) > 2 and s[0] in "[(" and bool(re.search(r"\d", s))
    return False


def is_nested(v):
    if isinstance(v, str):
        return v.strip().startswith(("[[", "[(", "(("))
    if isinstance(v, (list, tuple, np.ndarray)) and len(v):
        return isinstance(v[0], (list, tuple, np.ndarray, dict))
    return False


def detect_array_columns(df):
    out = []
    for c in df.columns:
        vals = df[c].dropna().head(20)
        if len(vals) and all(looks_array(v) for v in vals):
            out.append(c)
    return out


def build_colmap(path, split):
    cols = get_columns(path)
    pk = load_table(path, nrows=200)
    m = {role: guess_col(cols, role) for role in ROLE_CANDIDATES}
    arr = detect_array_columns(pk)
    for role in ARRAY_ROLES:                      # name matched but not array-valued -> not an array role
        if m[role] is not None and m[role] not in arr:
            m[role] = None
    for role in ROLE_CANDIDATES:                  # scalar roles may not point to array columns
        if role not in ARRAY_ROLES and m[role] in arr:
            m[role] = None
    unassigned = [c for c in arr if c not in {m[r] for r in ARRAY_ROLES}]
    for c in unassigned:
        v = pk[c].dropna().iloc[0]
        n = _norm(c)
        if is_nested(v) and m["peaks_col"] is None:
            m["peaks_col"] = c
        elif any(k in n for k in ("int", "abund", "height", "area")) and m["intensity_array_col"] is None:
            m["intensity_array_col"] = c
        elif m["mz_array_col"] is None:
            m["mz_array_col"] = c
    if m["peaks_col"] and m["mz_array_col"] and m["intensity_array_col"]:
        m["peaks_col"] = None
    m.update(COLMAP_OVERRIDES.get(split, {}))
    return m


assert FILES["train"] is not None, "No training file found — set FILE_OVERRIDES['train']"
assert FILES["test"] is not None, "No test file found — set FILE_OVERRIDES['test']"
COLMAP = {s: build_colmap(FILES[s], s) for s in ("train", "test")}

# Peak arrays may live in a separate file keyed by spectrum id.
PEAK_SRC = {}
for s in ("train", "test"):
    m = COLMAP[s]
    if m["peaks_col"] or (m["mz_array_col"] and m["intensity_array_col"]):
        PEAK_SRC[s] = {"path": FILES[s], "mz": m["mz_array_col"], "int": m["intensity_array_col"], "peaks": m["peaks_col"], "id_col": None}
        continue
    PEAK_SRC[s] = None
    for _, r in INVENTORY[INVENTORY.tabular].iterrows():
        p = Path(r.path)
        if str(p) in _used or s not in r.rel.lower():
            continue
        try:
            cm = build_colmap(p, s)
        except Exception:
            continue
        if (cm["peaks_col"] or (cm["mz_array_col"] and cm["intensity_array_col"])) and cm["spectrum_id_col"]:
            PEAK_SRC[s] = {"path": p, "mz": cm["mz_array_col"], "int": cm["intensity_array_col"], "peaks": cm["peaks_col"], "id_col": cm["spectrum_id_col"]}
            break

DATA_MAP = {
    "train_file": FILES["train"], "test_file": FILES["test"], "sample_submission_file": FILES["sample"],
    "candidate_library_file": FILES["candidate_library"],
    "spectrum_id_col": COLMAP["train"]["spectrum_id_col"],
    "molecule_id_col": COLMAP["test"]["molecule_id_col"] or COLMAP["train"]["molecule_id_col"],
    "smiles_col": COLMAP["train"]["smiles_col"],
    "precursor_mz_col": COLMAP["train"]["precursor_mz_col"],
    "mz_array_col": COLMAP["train"]["mz_array_col"],
    "intensity_array_col": COLMAP["train"]["intensity_array_col"],
    "peaks_col": COLMAP["train"]["peaks_col"],
    "adduct_col": COLMAP["train"]["adduct_col"],
    "ion_mode_col": COLMAP["train"]["ion_mode_col"],
    "collision_energy_col": COLMAP["train"]["collision_energy_col"],
    "instrument_col": COLMAP["train"]["instrument_col"],
    "source_col": COLMAP["train"]["source_col"],
    "ppm_error_col": COLMAP["train"]["ppm_error_col"],
}
for k, v in DATA_MAP.items():
    print(f"  {k:<24} {v}")
_diff = {r: (COLMAP['train'][r], COLMAP['test'][r]) for r in ROLE_CANDIDATES if COLMAP['train'][r] != COLMAP['test'][r]}
print("\nRoles whose column differs between train and test (train, test):")
for r, v in _diff.items():
    print(f"  {r:<24} {v}")
print("\nPeak-array source:", {s: (None if v is None else {k: str(x) for k, x in v.items()}) for s, v in PEAK_SRC.items()})

INPUT_FINGERPRINTS = {k: file_fingerprint(v) for k, v in FILES.items() if v is not None}
for s, v in PEAK_SRC.items():
    if v and str(v["path"]) not in {str(p) for p in FILES.values() if p}:
        INPUT_FINGERPRINTS[f"peaks_{s}"] = file_fingerprint(v["path"])

assert COLMAP["train"]["smiles_col"], "Training SMILES column not found — set COLMAP_OVERRIDES"
assert COLMAP["train"]["precursor_mz_col"] and COLMAP["test"]["precursor_mz_col"], "Precursor m/z column not found"
check("Dataset mapped", all(PEAK_SRC.values()) and FILES["sample"] is not None,
      "train/test/sample located, SMILES + precursor + peak arrays mapped")

### Safe cell-value helpers

Spectral cells may be Python lists, tuples, NumPy arrays, Arrow-converted object arrays, strings such as `"[1.0, 2.0]"`, or nested `[[mz, int], …]` / `[{mz:…, intensity:…}]` structures. All access goes through these two functions.

In [ ]:
def as_array(x, dtype=np.float64):
    """Coerce a scalar / list / tuple / ndarray / string cell into a flat 1-D float array."""
    if x is None:
        return np.empty(0, dtype)
    if isinstance(x, float) and np.isnan(x):
        return np.empty(0, dtype)
    if isinstance(x, np.ndarray):
        if x.dtype == object:
            return np.concatenate([as_array(v, dtype) for v in x]) if len(x) else np.empty(0, dtype)
        return x.astype(dtype, copy=False).ravel()
    if isinstance(x, (list, tuple)):
        try:
            return np.asarray(x, dtype=dtype).ravel()
        except (ValueError, TypeError):
            return np.concatenate([as_array(v, dtype) for v in x]) if len(x) else np.empty(0, dtype)
    if isinstance(x, (bytes, str)):
        s = x.decode() if isinstance(x, bytes) else x
        toks = [t for t in re.split(r"[\s,;:\[\]\(\)]+", s) if t]
        return np.asarray(toks, dtype=dtype) if toks else np.empty(0, dtype)
    return np.asarray([x], dtype=dtype)


def split_peaks(mz_cell=None, int_cell=None, peaks_cell=None):
    """Return (mz, intensity) float64 arrays from either two array cells or one combined peaks cell."""
    if peaks_cell is not None:
        v = peaks_cell
        if isinstance(v, (list, tuple, np.ndarray)) and len(v) and isinstance(v[0], dict):
            keys = list(v[0].keys())
            km = next(k for k in keys if "mz" in k.lower() or "mass" in k.lower())
            ki = next(k for k in keys if k != km)
            return (np.asarray([d[km] for d in v], float), np.asarray([d[ki] for d in v], float))
        flat = as_array(v)
        if isinstance(v, (list, tuple, np.ndarray)) and len(v) == 2 and len(flat) > 2 and not is_nested(v[:1]):
            half = len(flat) // 2                       # [[mz...], [int...]] layout
            return flat[:half], flat[half:]
        if len(flat) % 2:
            return flat[:0], flat[:0]
        pairs = flat.reshape(-1, 2)
        return pairs[:, 0], pairs[:, 1]
    return as_array(mz_cell), as_array(int_cell)


# unit checks of the helpers
_a, _b = split_peaks(peaks_cell=[[100.0, 5.0], [200.0, 10.0]])
assert np.allclose(_a, [100, 200]) and np.allclose(_b, [5, 10])
_a, _b = split_peaks("[100.1, 200.2]", np.array([1, 2]))
assert np.allclose(_a, [100.1, 200.2]) and np.allclose(_b, [1, 2])
_a, _b = split_peaks(peaks_cell=[{"mz": 50.0, "intensity": 3.0}])
assert _a[0] == 50 and _b[0] == 3
assert len(as_array(None)) == 0 and len(as_array(np.nan)) == 0 and as_array(3.5)[0] == 3.5
print("array helpers: PASS")

## 4. Determine the true data model

**QUESTION.** What is one row? Does one molecule own one spectrum or many, and what varies among its spectra? What is one prediction unit?

**WHY IT MATTERS.** If several spectra share one label, a spectrum-level random split leaks the answer, and the test's prediction unit (possibly a *group* of spectra) dictates how evidence is aggregated.

In [ ]:
STD_NAME = {
    "spectrum_id_col": "spectrum_id", "molecule_id_col": "molecule_id", "smiles_col": "smiles",
    "precursor_mz_col": "precursor_mz", "adduct_col": "adduct_raw", "ion_mode_col": "ion_mode_raw",
    "collision_energy_col": "ce_raw", "instrument_col": "instrument", "source_col": "source",
    "ppm_error_col": "ppm_provided", "charge_col": "charge_raw", "formula_col": "formula_provided",
    "inchikey_col": "inchikey_provided",
}


def _normalize_ion_mode(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "unknown"
    s = str(v).strip().lower()
    if s in ("positive", "pos", "+", "p", "1", "+1", "positive mode", "esi+", "[+]"):
        return "positive"
    if s in ("negative", "neg", "-", "n", "-1", "negative mode", "esi-", "[-]"):
        return "negative"
    return "other:" + s


def _cell_to_str(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (list, tuple, np.ndarray)):
        return "[" + ", ".join(f"{float(x):g}" for x in np.ravel(v)) + "]"
    return str(v).strip()


def load_meta(split):
    m, path = COLMAP[split], FILES[split]
    roles = {r: m[r] for r in STD_NAME if m.get(r)}
    df = load_table(path, columns=list(dict.fromkeys(roles.values())))
    df = pd.DataFrame({STD_NAME[r]: df[c].values for r, c in roles.items()})
    df["row_pos"] = np.arange(len(df), dtype=np.int64)
    if "spectrum_id" not in df:
        df["spectrum_id"] = [f"{split}_{i}" for i in range(len(df))]
    df["spectrum_id"] = df["spectrum_id"].astype(str)
    if "molecule_id" in df:
        df["molecule_id"] = df["molecule_id"].astype(str)
    pm = df["precursor_mz"]
    if not pd.api.types.is_numeric_dtype(pm):
        pm = pd.to_numeric(pm.astype(str).str.strip().str.split().str[0], errors="coerce")
    df["precursor_mz"] = pm.astype("float64")
    if "adduct_raw" in df:
        df["adduct"] = df["adduct_raw"].map(lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else re.sub(r"\s+", "", str(v)))
    if "ion_mode_raw" in df:
        u = {v: _normalize_ion_mode(v) for v in pd.unique(df["ion_mode_raw"])}
        df["ion_mode"] = df["ion_mode_raw"].map(u)
    if "ce_raw" in df:
        df["ce_raw"] = df["ce_raw"].map(_cell_to_str)
    if "ppm_provided" in df:
        df["ppm_provided"] = pd.to_numeric(df["ppm_provided"], errors="coerce")
    for c in ("adduct_raw", "adduct", "ion_mode_raw", "ion_mode", "instrument", "source", "ce_raw", "smiles", "molecule_id"):
        if c in df and df[c].nunique(dropna=True) < 0.5 * len(df) + 1:
            df[c] = df[c].astype("category")
    df["split"] = split
    return df


train_meta = load_meta("train")
test_meta = load_meta("test")
print(f"train_meta {train_meta.shape}, {train_meta.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"test_meta  {test_meta.shape}, {test_meta.memory_usage(deep=True).sum()/1e6:.1f} MB")


def peak_counts(split, meta):
    """Number of peaks per spectrum for ALL rows, streamed from Arrow list lengths (no peak values loaded)."""
    src = PEAK_SRC.get(split)
    if src is None or not _is_parquet_path(src["path"]):
        return None
    col = src["peaks"] or src["mz"]
    dset = pads.dataset(str(src["path"]), format="parquet")
    if not (pa.types.is_list(dset.schema.field(col).type) or pa.types.is_large_list(dset.schema.field(col).type)):
        return None
    cols = [col] + ([src["id_col"]] if src["id_col"] else [])
    lens, ids = [], []
    for b in dset.to_batches(columns=cols, batch_size=262_144):
        lens.append(pc.fill_null(pc.list_value_length(b.column(col)), 0).to_numpy(zero_copy_only=False))
        if src["id_col"]:
            ids.append(b.column(src["id_col"]).to_pandas().astype(str).values)
    lens = np.concatenate(lens).astype(np.int32)
    if src["peaks"] and not src["mz"]:
        pass  # nested [[mz, int], ...]: outer length == n_peaks
    if src["id_col"]:
        return pd.Series(lens, index=np.concatenate(ids)).reindex(meta["spectrum_id"].values).values
    return lens if len(lens) == len(meta) else None


for _df, _s in ((train_meta, "train"), (test_meta, "test")):
    _n = peak_counts(_s, _df)
    _df["n_peaks_all"] = _n if _n is not None else np.nan

# ---- prediction unit from the sample submission
sample_sub = load_table(FILES["sample"])
_test_orig_cols = {c for c in get_columns(FILES["test"])}
_common = [c for c in sample_sub.columns if c in _test_orig_cols]
SUB_ID_COL = COLMAP["test"]["molecule_id_col"] if COLMAP["test"]["molecule_id_col"] in _common else (_common[0] if _common else sample_sub.columns[0])
_role_of = {v: k for k, v in COLMAP["test"].items() if v}
PRED_UNIT_ROLE = _role_of.get(SUB_ID_COL, "unknown")
PRED_UNIT_STD = STD_NAME.get(PRED_UNIT_ROLE, None)
assert PRED_UNIT_STD in test_meta.columns, f"Cannot align sample-submission id column {SUB_ID_COL!r} to test metadata"
test_meta["unit_id"] = test_meta[PRED_UNIT_STD].astype(str)

# ---- data-model facts
spec_id_unique_train = train_meta["spectrum_id"].is_unique
spec_id_unique_test = test_meta["spectrum_id"].is_unique
spectra_per_smiles = train_meta.groupby("smiles", observed=True).size()
spectra_per_unit = test_meta.groupby("unit_id", observed=True).size()
train_has_molid = "molecule_id" in train_meta
if train_has_molid:
    _mm = train_meta.groupby("molecule_id", observed=True)["smiles"].nunique()
    print(f"train molecule_id → n distinct SMILES: max={_mm.max()}  (1 means molecule_id is a label-level id)")

ACQ_COLS = [c for c in ("adduct", "ion_mode", "ce_raw", "instrument", "source") if c in train_meta]
_multi = train_meta[train_meta["smiles"].map(spectra_per_smiles).astype(float) > 1]
VARIES = {}
for c in ACQ_COLS:
    nun = _multi.groupby("smiles", observed=True)[c].nunique()
    VARIES[c] = float((nun > 1).mean()) if len(nun) else 0.0

n_unique_smiles = train_meta["smiles"].nunique()
DATA_MODEL = {
    "OBSERVATION UNIT": "one MS/MS spectrum (one row)" if spec_id_unique_train else "row (spectrum_id NOT unique — investigate)",
    "PREDICTION UNIT": f"one test '{SUB_ID_COL}' ({PRED_UNIT_ROLE}); spectra per unit: median {spectra_per_unit.median():.0f}, max {spectra_per_unit.max()}",
    "STRUCTURE LABEL": f"SMILES column '{COLMAP['train']['smiles_col']}' (evaluated by connectivity, §6)",
    "NUMBER OF TRAIN SPECTRA": len(train_meta),
    "NUMBER OF TEST SPECTRA": len(test_meta),
    "NUMBER OF UNIQUE TRAIN SMILES (raw)": n_unique_smiles,
    "NUMBER OF UNIQUE TRAIN MOLECULES": "see §7 (connectivity level)",
    "NUMBER OF TEST PREDICTION UNITS": test_meta["unit_id"].nunique(),
    "sample submission rows": len(sample_sub),
    "train spectra per raw SMILES (median / max)": f"{spectra_per_smiles.median():.0f} / {spectra_per_smiles.max()}",
    "multi-spectrum molecules where each field varies": {k: f"{v:.1%}" for k, v in VARIES.items()},
}
assert spec_id_unique_train and spec_id_unique_test, "spectrum ids are not unique — the observation unit is not a spectrum"
assert set(sample_sub[SUB_ID_COL].astype(str)) == set(test_meta["unit_id"]), "sample-submission ids ≠ test prediction units"
report("§4 data model", DATA_MODEL,
       "One row = one spectrum; many spectra can share one structure → all splits must be grouped by structure. "
       "The test prediction unit is the sample-submission id; its spectra must be aggregated into one ranked list.")

## 5–6. Robust molecular structure table and the competition connectivity key

**QUESTION.** How does the competition decide that two SMILES are the "same answer", and can we reproduce it?

**ANALYSIS.** The competition compares *connectivity*: `SMILES → RDKit mol → tautomer canonicalisation → InChIKey → first 14 characters`. The first InChIKey block hashes the skeleton + hydrogen layer only, so stereochemistry and isotopes are ignored; tautomer canonicalisation additionally merges tautomers that standard InChI's mobile-H does not. `raw_smiles`, `canonical_smiles` and `connectivity_key` are stored side by side and the raw label is never overwritten.

The function is unit-tested (§6 tests A–D) **before** any structure table or fold is built. The structure table is computed in parallel (joblib/loky) and cached with provenance.

In [ ]:
def _get_tautomer_enumerator():
    # Stored on the (picklable-by-reference) module so each worker process builds its own instance.
    te = getattr(rdMolStandardize, "_casmi_te", None)
    if te is None:
        te = rdMolStandardize.TautomerEnumerator()
        if CONFIG.get("tautomer_max_tautomers"):
            te.SetMaxTautomers(int(CONFIG["tautomer_max_tautomers"]))
        rdMolStandardize._casmi_te = te
    return te


def _mol_from_smiles(smiles):
    if smiles is None or (isinstance(smiles, float) and np.isnan(smiles)):
        return None
    s = str(smiles).strip()
    if not s:
        return None
    with rdkit_quiet():
        return Chem.MolFromSmiles(s)


def _connectivity_from_mol(mol):
    try:
        with rdkit_quiet():
            mol_t = _get_tautomer_enumerator().Canonicalize(mol)
    except Exception:
        mol_t = mol
    with rdkit_quiet():
        ik = Chem.MolToInchiKey(mol_t) or Chem.MolToInchiKey(mol)
    return ik[:14] if ik else None


def competition_connectivity_key(smiles):
    """SMILES → RDKit mol → tautomer canonical → InChIKey → first block. Returns None if invalid."""
    mol = _mol_from_smiles(smiles)
    return None if mol is None else _connectivity_from_mol(mol)


# ---------------- §6 unit tests (run before any table / fold) ----------------
def _same(a, b):
    ka, kb = competition_connectivity_key(a), competition_connectivity_key(b)
    return ka is not None and ka == kb


CONN_TESTS = {
    "A1 stereo: L- vs D-alanine": _same("C[C@@H](C(=O)O)N", "C[C@H](C(=O)O)N"),
    "A2 stereo vs no stereo": _same("C[C@@H](C(=O)O)N", "CC(C(=O)O)N"),
    "A3 E vs Z but-2-ene": _same("C/C=C/C", "C/C=C\\C"),
    "B1 tautomer: 2-hydroxypyridine vs 2-pyridone": _same("Oc1ccccn1", "O=c1cccc[nH]1"),
    "B2 tautomer: keto vs enol acetone": _same("CC(=O)C", "CC(O)=C"),
    "B3 same molecule, different SMILES writing": _same("OCC", "C(O)C"),
    "C1 isomers: ethanol vs dimethyl ether": not _same("CCO", "COC"),
    "C2 isomers: o- vs p-cresol": not _same("Cc1ccccc1O", "Cc1ccc(O)cc1"),
    "C3 isomers: leucine vs isoleucine": not _same("CC(C)CC(N)C(=O)O", "CCC(C)C(N)C(=O)O"),
    "D1 invalid SMILES → None": competition_connectivity_key("not_a_smiles(") is None,
    "D2 empty / None / NaN → None": all(competition_connectivity_key(x) is None for x in ("", None, float("nan"))),
    "E key is 14 upper-case letters": bool(re.fullmatch(r"[A-Z]{14}", competition_connectivity_key("CCO") or "")),
}
for k, v in CONN_TESTS.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
check("Competition connectivity validated", all(CONN_TESTS.values()), f"{sum(CONN_TESTS.values())}/{len(CONN_TESTS)} tests")

In [ ]:
HALOGENS = {"F", "Cl", "Br", "I"}
STRUCT_COLS = ["raw_smiles", "valid", "error", "canonical_smiles", "inchikey", "inchikey14", "connectivity_key",
               "molecular_formula", "exact_mass", "molecular_weight", "n_fragments",
               "num_atoms", "num_heavy_atoms", "num_c", "num_n", "num_o", "num_s", "num_p", "num_halogen",
               "num_rings", "num_aromatic_rings", "fraction_aromatic_atoms", "num_heteroatoms",
               "num_rotatable_bonds", "formal_charge", "h_bond_donors", "h_bond_acceptors", "tpsa", "logp",
               "murcko_scaffold", "scaffold_smiles"]


def compute_molecule_descriptors(smiles):
    """One structure record. Never raises: invalid input gives valid=False and an error message."""
    rec = dict.fromkeys(STRUCT_COLS)
    rec["raw_smiles"] = smiles
    rec["valid"] = False
    try:
        mol = _mol_from_smiles(smiles)
        if mol is None:
            rec["error"] = "rdkit_parse_failed"
            return rec
        with rdkit_quiet():
            rec["canonical_smiles"] = Chem.MolToSmiles(mol)
            ik = Chem.MolToInchiKey(mol)
        rec["inchikey"] = ik or None
        rec["inchikey14"] = ik[:14] if ik else None
        rec["connectivity_key"] = _connectivity_from_mol(mol)
        rec["molecular_formula"] = rdMolDescriptors.CalcMolFormula(mol)
        rec["exact_mass"] = rdMolDescriptors.CalcExactMolWt(mol)
        rec["molecular_weight"] = Descriptors.MolWt(mol)
        rec["n_fragments"] = len(Chem.GetMolFrags(mol))
        cnt = Counter()
        n_h = 0
        arom = 0
        for a in mol.GetAtoms():
            cnt[a.GetSymbol()] += 1
            n_h += a.GetTotalNumHs()
            arom += a.GetIsAromatic()
        heavy = mol.GetNumHeavyAtoms()
        rec.update(num_atoms=mol.GetNumAtoms() + n_h, num_heavy_atoms=heavy,
                   num_c=cnt.get("C", 0), num_n=cnt.get("N", 0), num_o=cnt.get("O", 0),
                   num_s=cnt.get("S", 0), num_p=cnt.get("P", 0),
                   num_halogen=sum(cnt.get(h, 0) for h in HALOGENS),
                   num_rings=rdMolDescriptors.CalcNumRings(mol),
                   num_aromatic_rings=rdMolDescriptors.CalcNumAromaticRings(mol),
                   fraction_aromatic_atoms=arom / heavy if heavy else 0.0,
                   num_heteroatoms=rdMolDescriptors.CalcNumHeteroatoms(mol),
                   num_rotatable_bonds=rdMolDescriptors.CalcNumRotatableBonds(mol),
                   formal_charge=Chem.GetFormalCharge(mol),
                   h_bond_donors=rdMolDescriptors.CalcNumHBD(mol),
                   h_bond_acceptors=rdMolDescriptors.CalcNumHBA(mol),
                   tpsa=rdMolDescriptors.CalcTPSA(mol), logp=Crippen.MolLogP(mol))
        try:
            core = MurckoScaffold.GetScaffoldForMol(mol)
            rec["murcko_scaffold"] = Chem.MolToSmiles(core) if core.GetNumAtoms() else ""
            rec["scaffold_smiles"] = Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(core)) if core.GetNumAtoms() else ""
        except Exception as ex:
            rec["murcko_scaffold"], rec["scaffold_smiles"] = None, None
            rec["error"] = f"scaffold:{type(ex).__name__}"
        rec["valid"] = True
    except Exception as ex:
        rec["error"] = f"{type(ex).__name__}: {ex}"[:200]
    return rec


def _records_chunk(smiles_chunk):
    return [compute_molecule_descriptors(s) for s in smiles_chunk]


def build_structure_table(smiles_list, n_jobs):
    smiles_list = list(smiles_list)
    chunks = [smiles_list[i:i + 500] for i in range(0, len(smiles_list), 500)]
    if n_jobs > 1 and len(smiles_list) > 2000 and OPTIONAL.get("joblib"):
        from joblib import Parallel, delayed
        parts = Parallel(n_jobs=n_jobs, backend="loky")(delayed(_records_chunk)(c) for c in chunks)
    else:
        parts = [_records_chunk(c) for c in chunks]
    return pd.DataFrame([r for p in parts for r in p], columns=STRUCT_COLS)


_raw_smiles = pd.Series(train_meta["smiles"].dropna().astype(str).unique())
_struct_cfg = {"PIPELINE_VERSION": PIPELINE_VERSION, "tautomer_max_tautomers": CONFIG["tautomer_max_tautomers"],
               "smiles_hash": hashlib.sha256("\n".join(sorted(_raw_smiles)).encode()).hexdigest()[:16]}
t0 = time.time()
if artifact_is_fresh("structure_table", _struct_cfg):
    struct_df = pd.read_parquet(CACHE_ROOT / "structure_table.parquet")
    print(f"structure_table: loaded fresh cache ({len(struct_df):,} rows)")
else:
    struct_df = build_structure_table(_raw_smiles, CONFIG["n_jobs"])
    save_artifact(struct_df, "structure_table", config=_struct_cfg,
                  extra={"description": "one row per unique raw training SMILES"})
    print(f"structure_table: computed {len(struct_df):,} rows in {time.time()-t0:.0f}s")

n_null_label_spectra = int(train_meta["smiles"].isna().sum())
n_invalid = int((~struct_df["valid"]).sum())
_spec_per_raw = train_meta["smiles"].astype(object).value_counts()
struct_df["n_spectra"] = struct_df["raw_smiles"].map(_spec_per_raw).fillna(0).astype(int)
invalid_tbl = struct_df.loc[~struct_df["valid"], ["raw_smiles", "error", "n_spectra"]]
report("§5 structure table", {
    "unique raw SMILES": len(struct_df),
    "invalid SMILES (kept, flagged)": f"{n_invalid} ({n_invalid/len(struct_df):.3%}), affecting {int(invalid_tbl.n_spectra.sum())} spectra",
    "spectra with missing label": n_null_label_spectra,
    "multi-fragment SMILES (salts/mixtures)": int((struct_df.n_fragments.fillna(1) > 1).sum()),
    "charged structures (formal charge ≠ 0)": int((struct_df.formal_charge.fillna(0) != 0).sum()),
}, "Invalid structures are retained with valid=False; their spectra get connectivity_key=None and fold=-1 (never silently dropped).")
if len(invalid_tbl):
    display(invalid_tbl.head(10))  # noqa: F821

## 7. Evaluation-equivalent structures

**QUESTION.** How many distinct *answers* exist once representation differences are collapsed?

**WHY IT MATTERS.** Several raw SMILES mapping to one connectivity are *multiple representations of one answer*, not label collisions. Candidate lists must be deduplicated by connectivity, otherwise duplicates waste ranks.

In [ ]:
v = struct_df[struct_df.valid]
raw_per_conn = v.groupby("connectivity_key").size()
_ex = raw_per_conn[raw_per_conn > 1].sort_values(ascending=False).head(3).index
_examples = v[v.connectivity_key.isin(_ex)][["connectivity_key", "raw_smiles"]].sort_values("connectivity_key")
report("§7 equivalence", {
    "unique raw SMILES": v.raw_smiles.nunique(),
    "unique canonical SMILES": v.canonical_smiles.nunique(),
    "unique full InChIKeys": v.inchikey.nunique(),
    "unique InChIKey14 (no tautomer step)": v.inchikey14.nunique(),
    "unique connectivity keys": v.connectivity_key.nunique(),
    "connectivities with >1 raw SMILES": f"{(raw_per_conn > 1).sum()} ({(raw_per_conn > 1).mean():.2%})",
    "raw SMILES per connectivity (max)": int(raw_per_conn.max()),
    "extra merges from tautomer canonicalisation": v.inchikey14.nunique() - v.connectivity_key.nunique(),
}, "Unit of identity = connectivity_key. Ranked candidate lists will be deduplicated by connectivity_key, keeping the first occurrence.")
display(_examples.head(9))  # noqa: F821

## 8. Molecule-level analytical table (`molecule_df`)

One row = one competition-equivalent connectivity. The representative SMILES is the raw variant with the most spectra (ties broken alphabetically, deterministic).

In [ ]:
DESC_COLS = ["exact_mass", "molecular_weight", "num_atoms", "num_heavy_atoms", "num_c", "num_n", "num_o", "num_s", "num_p",
             "num_halogen", "num_rings", "num_aromatic_rings", "fraction_aromatic_atoms", "num_heteroatoms",
             "num_rotatable_bonds", "formal_charge", "h_bond_donors", "h_bond_acceptors", "tpsa", "logp"]
_v = struct_df[struct_df.valid].sort_values(["connectivity_key", "n_spectra", "raw_smiles"], ascending=[True, False, True])
_rep = _v.drop_duplicates("connectivity_key", keep="first")
_agg = _v.groupby("connectivity_key").agg(n_raw_smiles_variants=("raw_smiles", "size"),
                                         n_spectra=("n_spectra", "sum"),
                                         n_formulas=("molecular_formula", "nunique"))
molecule_df = (_rep[["connectivity_key", "canonical_smiles", "raw_smiles", "molecular_formula", "murcko_scaffold", "scaffold_smiles", "n_fragments"] + DESC_COLS]
               .rename(columns={"canonical_smiles": "representative_smiles", "raw_smiles": "representative_raw_smiles"})
               .merge(_agg, on="connectivity_key").reset_index(drop=True))
for c in DESC_COLS:
    if molecule_df[c].dtype.kind == "f":
        molecule_df[c] = molecule_df[c].astype("float64" if c in ("exact_mass", "molecular_weight") else "float32")
N_MOL = len(molecule_df)
_mf = int((molecule_df.n_formulas > 1).sum())
report("§8 molecule table", {"molecules (connectivities)": N_MOL,
                              "connectivities whose variants disagree on formula": _mf},
       "molecule_df is the canonical molecular table (saved in §37 as molecule_metadata.parquet). "
       "Formula disagreement within a connectivity can only arise from charge/protonation variants; it is reported, not hidden.")

# map spectra → connectivity / exact mass of THEIR raw SMILES (tautomers/stereo share mass)
_key_map = struct_df.set_index("raw_smiles")["connectivity_key"]
_mass_map = struct_df.set_index("raw_smiles")["exact_mass"]
train_meta["connectivity_key"] = train_meta["smiles"].astype(object).map(_key_map).astype("category")
train_meta["exact_mass"] = train_meta["smiles"].astype(object).map(_mass_map).astype("float64")
print(f"spectra without a valid connectivity: {train_meta.connectivity_key.isna().sum():,}")
DATA_MODEL_CONN = {"NUMBER OF UNIQUE TRAIN MOLECULES": N_MOL}
print("OBSERVATION UNIT =", DATA_MODEL["OBSERVATION UNIT"])
print("PREDICTION UNIT =", DATA_MODEL["PREDICTION UNIT"])
print("NUMBER OF UNIQUE TRAIN MOLECULES (connectivity) =", N_MOL)

## 9. Spectra per molecule

**QUESTION.** How much repeated evidence exists per molecule, and how diverse are the acquisitions behind it?

**WHY IT MATTERS.** Molecules with many spectra dominate spectrum-weighted losses and metrics; singletons offer no within-molecule variability. This drives molecule-level weighting and aggregation later.

In [ ]:
_g = train_meta.dropna(subset=["connectivity_key"]).groupby("connectivity_key", observed=True)
per_mol = _g.size().rename("n_spectra").to_frame()
for c, name in (("adduct", "n_adducts"), ("ion_mode", "n_ion_modes"), ("instrument", "n_instruments"),
                ("ce_raw", "n_collision_conditions"), ("source", "n_sources")):
    if c in train_meta:
        per_mol[name] = _g[c].nunique()
spm = per_mol["n_spectra"]
report("§9 spectra per molecule", {**{k: round(float(v), 2) for k, v in qsummary(spm, (0, .5, .9, .95, .99, 1)).items()},
                                    "mean": float(spm.mean()),
                                    "singleton molecules": f"{(spm == 1).mean():.1%}",
                                    "share of spectra from top-1% molecules": f"{spm.sort_values(ascending=False).head(max(1, len(spm)//100)).sum()/spm.sum():.1%}"},
       "Metrics will be reported per molecule (each validation connectivity counts once) in addition to per spectrum, "
       "so heavily-replicated molecules cannot dominate model selection.")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(np.log10(spm), bins=50, color="#4472c4")
axes[0].set_xlabel("log10(spectra per molecule)"); axes[0].set_ylabel("molecules"); axes[0].set_title("Spectra per connectivity")
_cols = [c for c in per_mol.columns if c != "n_spectra"]
if _cols:
    per_mol.loc[spm > 1, _cols].apply(lambda s: s.clip(upper=6).value_counts().sort_index()).fillna(0).plot.bar(ax=axes[1])
    axes[1].set_title("Distinct acquisition values per multi-spectrum molecule (clipped at 6)"); axes[1].set_xlabel("distinct values")
savefig(fig, "09_spectra_per_molecule")
for label, sel in (("1 spectrum", spm == 1), ("2–5 spectra", spm.between(2, 5)), ("many spectra (top)", spm >= spm.quantile(0.99))):
    ids = per_mol[sel].sample(min(3, int(sel.sum())), random_state=SEED).index if sel.sum() else []
    print(f"\n{label}:")
    if len(ids):
        display(per_mol.loc[ids].join(molecule_df.set_index("connectivity_key")[["representative_smiles", "exact_mass"]]))  # noqa: F821

## 10. Precursor m/z quality audit

**QUESTION.** Is the precursor m/z physically plausible everywhere, and does a handful of extreme values distort the picture?

**ANALYSIS.** Quantiles first; then an explicit table of suspicious values (non-finite, ≤ 0, or > 6 robust SDs from the median in log10 space). Figure A shows the full range (log-count axis); Figure B clips to a percentile window **for display only** and states how many observations fall outside.

In [ ]:
PM_Q = (0, 0.0001, 0.001, 0.01, 0.5, 0.99, 0.999, 0.9999, 1)
PM_NAMES = ["min", "p0.01", "p0.1", "p1", "median", "p99", "p99.9", "p99.99", "max"]
pm_table = pd.DataFrame({s: qsummary(d["precursor_mz"], PM_Q, PM_NAMES) for s, d in (("train", train_meta), ("test", test_meta))}).T
display(pm_table.round(4))  # noqa: F821


def flag_precursor(df):
    x = df["precursor_mz"].to_numpy()
    lx = np.log10(np.where(x > 0, x, np.nan))
    med = np.nanmedian(lx)
    mad = 1.4826 * np.nanmedian(np.abs(lx - med))
    rz = np.abs(lx - med) / (mad if mad > 0 else 1)
    reason = np.where(~np.isfinite(x), "non-finite", np.where(x <= 0, "non-positive", np.where(rz > 6, "robust-z>6", "")))
    return pd.Series(reason, index=df.index), rz


suspicious = []
for s, d in (("train", train_meta), ("test", test_meta)):
    reason, rz = flag_precursor(d)
    d["precursor_flag"] = pd.Categorical(reason)
    t = d.loc[reason != "", ["spectrum_id", "precursor_mz"] + [c for c in ("adduct", "source", "smiles") if c in d]].copy()
    t["reason"], t["robust_z"], t["split"] = reason[reason != ""], rz[reason != ""], s
    suspicious.append(t)
suspicious = pd.concat(suspicious, ignore_index=True).sort_values("robust_z", ascending=False)
print(f"suspicious precursor values: {len(suspicious)}")
display(suspicious.head(15))  # noqa: F821

lo, hi = np.nanquantile(pd.concat([train_meta.precursor_mz, test_meta.precursor_mz]), [0.001, 0.999])
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for s, d, col in (("train", train_meta, "#4472c4"), ("test", test_meta, "#ed7d31")):
    x = d.precursor_mz[np.isfinite(d.precursor_mz)]
    axes[0].hist(x, bins=200, alpha=0.6, color=col, label=s, density=True, log=True)
    axes[1].hist(x[(x >= lo) & (x <= hi)], bins=120, alpha=0.6, color=col, label=s, density=True)
axes[0].set_title("A. Full range (log density)"); axes[0].set_xlabel("precursor m/z"); axes[0].legend()
n_out = {s: int(((d.precursor_mz < lo) | (d.precursor_mz > hi)).sum()) for s, d in (("train", train_meta), ("test", test_meta))}
axes[1].set_title(f"B. p0.1–p99.9 window [{lo:.0f}, {hi:.0f}] — outside: {n_out}"); axes[1].set_xlabel("precursor m/z"); axes[1].legend()
savefig(fig, "10_precursor_distribution")
if "source" in train_meta:
    top_src = train_meta.source.value_counts().head(8).index
    fig, ax = plt.subplots(figsize=(13, 3.2))
    for s in top_src:
        x = train_meta.loc[train_meta.source == s, "precursor_mz"]
        ax.hist(x[(x >= lo) & (x <= hi)], bins=100, histtype="step", density=True, label=f"train:{s}")
    x = test_meta.precursor_mz
    ax.hist(x[(x >= lo) & (x <= hi)], bins=100, histtype="stepfilled", alpha=0.25, color="k", density=True, label="test")
    ax.legend(fontsize=7, ncol=3); ax.set_title("Precursor m/z by training source vs test (display window)")
    savefig(fig, "10_precursor_by_source")
report("§10 precursor audit", {"suspicious train": int((suspicious.split == "train").sum()),
                                "suspicious test": int((suspicious.split == "test").sum()),
                                "display window": f"[{lo:.1f}, {hi:.1f}]", "outside window": n_out},
       "Suspicious precursors are FLAGGED (precursor_flag column), not removed. Mass-based candidate filtering "
       "must treat flagged test spectra with a fallback (wider window / other spectra of the same unit).")

## 11. Adduct analysis

**QUESTION.** Which adducts occur in train and test, and are any test adducts unseen in training?

**WHY IT MATTERS.** Adduct determines neutral mass and fragmentation behaviour. A test-only adduct has no training evidence and must have an explicit fallback.

In [ ]:
if "adduct" in train_meta:
    a_tr = train_meta.adduct.astype(object).fillna("<missing>").value_counts()
    a_te = test_meta.adduct.astype(object).fillna("<missing>").value_counts() if "adduct" in test_meta else pd.Series(dtype=int)
    adduct_tbl = pd.DataFrame({"train_n": a_tr, "test_n": a_te}).fillna(0).astype(int)
    adduct_tbl["train_pct"] = 100 * adduct_tbl.train_n / adduct_tbl.train_n.sum()
    adduct_tbl["test_pct"] = 100 * adduct_tbl.test_n / max(1, adduct_tbl.test_n.sum())
    adduct_tbl = adduct_tbl.sort_values(["test_n", "train_n"], ascending=False)
    train_only = adduct_tbl[(adduct_tbl.train_n > 0) & (adduct_tbl.test_n == 0)].index.tolist()
    test_only = adduct_tbl[(adduct_tbl.test_n > 0) & (adduct_tbl.train_n == 0)].index.tolist()
    rare = adduct_tbl[(adduct_tbl.train_pct < 100 * CONFIG["rare_adduct_frac"]) & (adduct_tbl.train_n > 0)].index.tolist()
    display(adduct_tbl.head(25).round(3))  # noqa: F821
    union = list(dict.fromkeys(a_tr.head(12).index.tolist() + a_te.head(12).index.tolist()))
    fig, ax = plt.subplots(figsize=(13, 3.4))
    xx = np.arange(len(union))
    ax.bar(xx - 0.2, adduct_tbl.loc[union, "train_pct"], 0.4, label="train")
    ax.bar(xx + 0.2, adduct_tbl.loc[union, "test_pct"], 0.4, label="test")
    ax.set_xticks(xx); ax.set_xticklabels(union, rotation=45, ha="right", fontsize=8); ax.set_ylabel("% of spectra"); ax.set_yscale("log")
    ax.legend(); ax.set_title("Adducts — union of top train and top test categories")
    savefig(fig, "11_adducts")
    report("§11 adducts", {"distinct train adducts": int((adduct_tbl.train_n > 0).sum()),
                           "distinct test adducts": int((adduct_tbl.test_n > 0).sum()),
                           "train-only adducts": len(train_only), "test-only adducts": test_only,
                           f"rare train adducts (<{CONFIG['rare_adduct_frac']:.1%})": len(rare),
                           "test spectra with test-only adduct": int(adduct_tbl.loc[test_only, "test_n"].sum()) if test_only else 0},
           "Adduct is an explicit input everywhere; test-only adducts need the fallback defined in §40.")
else:
    report("§11 adducts", "No adduct column exists. Neutral mass cannot be reconstructed from precursor m/z alone; "
                          "candidate generation must enumerate plausible adducts per ion mode.")

## 12. General adduct parser

**ANALYSIS.** Adducts are parsed from their notation (`[nM ± species …]z±`) rather than looked up in a fixed list, so every adduct in the data is attempted. Species are either chemical formulae (masses from monoisotopic element masses) or documented abbreviations (FA, Hac, ACN, …). Electron mass is handled explicitly:

`ion_mass = n·M + Σ(±species) − z·mₑ`, `m/z = ion_mass / |z|`, and inversely `M = (m/z·|z| − Σ(±species) + z·mₑ) / n`.

Anything that cannot be parsed unambiguously — unknown abbreviation, missing charge that cannot be inferred from a single simple ion, `M` absent — is flagged **unsupported** with a reason. Nothing is approximated silently.

In [ ]:
ELECTRON_MASS = 0.000548579909
PROTON_MASS = 1.007276466812
ELEMENT_MASS = {"H": 1.00782503207, "D": 2.01410177812, "C": 12.0, "N": 14.0030740048, "O": 15.99491461956,
                "F": 18.99840322, "Na": 22.9897692809, "Mg": 23.985041700, "P": 30.97376163, "S": 31.97207100,
                "Cl": 34.96885268, "K": 38.96370668, "Ca": 39.96259098, "Fe": 55.9349375, "Cu": 62.9295975,
                "Zn": 63.9291422, "Br": 78.9183371, "Li": 7.01600455, "I": 126.904473, "Ag": 106.905097, "Cs": 132.905451929}
SPECIES_ALIASES = {"fa": "CH2O2", "hcooh": "CH2O2", "formate": "CHO2", "hac": "C2H4O2", "acoh": "C2H4O2",
                   "ch3cooh": "C2H4O2", "oac": "C2H3O2", "acetate": "C2H3O2", "acn": "C2H3N", "mecn": "C2H3N",
                   "meoh": "CH4O", "etoh": "C2H6O", "dmso": "C2H6OS", "isoprop": "C3H8O", "ipa": "C3H8O", "tfa": "C2HF3O2"}
SIMPLE_CATIONS = {"H", "Na", "K", "Li", "NH4", "Cs", "Ag"}
SIMPLE_ANIONS_ADDED = {"Cl", "Br", "I", "CHO2", "HCOO", "C2H3O2", "CH3COO", "FA", "Hac", "OAc"}
ADDUCT_RE = re.compile(r"^\[?(?P<n>\d*)M(?P<terms>(?:[+-]\d*[A-Za-z][A-Za-z0-9]*)*)\]?(?P<charge>\d*[+-]*)$")
TERM_RE = re.compile(r"([+-])(\d*)([A-Za-z][A-Za-z0-9]*)")


def formula_mass(formula):
    toks = re.findall(r"([A-Z][a-z]?)(\d*)", formula)
    if "".join(e + c for e, c in toks) != formula:
        return None
    total = 0.0
    for e, c in toks:
        if e not in ELEMENT_MASS:
            return None
        total += ELEMENT_MASS[e] * (int(c) if c else 1)
    return total


def species_mass(sp_):
    f = SPECIES_ALIASES.get(sp_.lower())
    return formula_mass(f) if f else formula_mass(sp_)


def parse_adduct(adduct):
    out = {"adduct": adduct, "n": np.nan, "z": np.nan, "mass_shift": np.nan, "supported": False, "reason": "", "charge_inferred": False}
    if adduct is None or (isinstance(adduct, float) and np.isnan(adduct)):
        out["reason"] = "missing"
        return out
    s = re.sub(r"\s+", "", str(adduct)).rstrip(".*•")
    m = ADDUCT_RE.match(s)
    if not m:
        out["reason"] = "unparseable notation"
        return out
    n = int(m["n"]) if m["n"] else 1
    shift, terms = 0.0, TERM_RE.findall(m["terms"])
    for sign, mult, spc in terms:
        ms = species_mass(spc)
        if ms is None:
            out["reason"] = f"unknown species '{spc}'"
            return out
        shift += (1 if sign == "+" else -1) * (int(mult) if mult else 1) * ms
    ch = m["charge"]
    if ch:
        cm = re.fullmatch(r"(\d*)([+-]+)", ch)
        if not cm or len(set(cm[2])) != 1 or (cm[1] and len(cm[2]) > 1):
            out["reason"] = f"bad charge '{ch}'"
            return out
        z = (int(cm[1]) if cm[1] else len(cm[2])) * (1 if cm[2][0] == "+" else -1)
    else:
        if len(terms) == 1 and terms[0][0] == "+" and terms[0][2] in SIMPLE_CATIONS:
            z = int(terms[0][1] or 1)
        elif len(terms) == 1 and terms[0][0] == "-" and terms[0][2] == "H":
            z = -int(terms[0][1] or 1)
        elif len(terms) == 1 and terms[0][0] == "+" and terms[0][2] in SIMPLE_ANIONS_ADDED:
            z = -1
        else:
            out["reason"] = "charge missing and not inferable"
            return out
        out["charge_inferred"] = True
    if z == 0:
        out["reason"] = "zero charge"
        return out
    out.update(n=n, z=z, mass_shift=shift, supported=True, reason="ok")
    return out


def precursor_from_neutral_mass(neutral_mass, adduct):
    p = parse_adduct(adduct) if isinstance(adduct, str) else adduct
    if not p["supported"]:
        return np.nan
    return (p["n"] * np.asarray(neutral_mass, float) + p["mass_shift"] - p["z"] * ELECTRON_MASS) / abs(p["z"])


def neutral_mass_from_precursor(precursor_mz, adduct):
    p = parse_adduct(adduct) if isinstance(adduct, str) else adduct
    if not p["supported"]:
        return np.nan
    return (np.asarray(precursor_mz, float) * abs(p["z"]) - p["mass_shift"] + p["z"] * ELECTRON_MASS) / p["n"]


def adduct_arrays(adduct_series):
    """Vectorised parse: returns DataFrame (n, z, mass_shift, supported) aligned to the series."""
    uniq = pd.unique(adduct_series.astype(object))
    tbl = pd.DataFrame([parse_adduct(a) for a in uniq])
    tbl.index = pd.Index(uniq, dtype=object)
    return tbl.reindex(adduct_series.astype(object).values).reset_index(drop=True)


if "adduct" in train_meta:
    ADDUCT_TABLE = pd.DataFrame([parse_adduct(a) for a in adduct_tbl.index if a != "<missing>"])
    ADDUCT_TABLE = ADDUCT_TABLE.merge(adduct_tbl[["train_n", "test_n"]], left_on="adduct", right_index=True)
    display(ADDUCT_TABLE.sort_values("train_n", ascending=False).head(30))  # noqa: F821
    unsup = ADDUCT_TABLE[~ADDUCT_TABLE.supported]
    report("§12 adduct parser", {"adduct strings": len(ADDUCT_TABLE), "supported": int(ADDUCT_TABLE.supported.sum()),
                                 "unsupported": unsup[["adduct", "reason", "train_n", "test_n"]].to_dict("records")[:15],
                                 "train spectra with unsupported adduct": int(unsup.train_n.sum()),
                                 "test spectra with unsupported adduct": int(unsup.test_n.sum()),
                                 "charge inferred (no explicit charge)": int(ADDUCT_TABLE.charge_inferred.sum())},
           "Unsupported adducts get neutral_mass = NaN and are handled by the §40 fallback (enumerate plausible adducts for the ion mode).")
    for d in (train_meta, test_meta):
        if "adduct" in d:
            aa = adduct_arrays(d["adduct"])
            d["adduct_supported"] = aa["supported"].astype(bool).values
            d["adduct_z"] = aa["z"].astype("float32").values
            d["neutral_mass_obs"] = ((d.precursor_mz.values * np.abs(aa.z.values) - aa.mass_shift.values + aa.z.values * ELECTRON_MASS) / aa.n.values)
            if "ion_mode" not in d:
                d["ion_mode"] = pd.Categorical(np.where(aa.z > 0, "positive", np.where(aa.z < 0, "negative", "unknown")))
                d["ion_mode_derived_from_adduct"] = True
else:
    ADDUCT_TABLE = pd.DataFrame(columns=["adduct", "supported"])
    for d in (train_meta, test_meta):
        d["adduct_supported"] = False
        d["neutral_mass_obs"] = np.nan

## 13. Unit-test adduct algebra (pure mathematics, no instrument error)

Synthetic neutral masses → precursor m/z → reconstructed neutral mass, for every supported adduct in the data plus reference adducts. Reference values anchor the physics (e.g. `[M+H]+` of M = 100 must be 101.0072765).

In [ ]:
REF_ADDUCTS = ["[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+K]+", "[M+H-H2O]+", "[2M+H]+", "[M+2H]2+",
               "[M+Cl]-", "[M+FA-H]-", "[M]+", "[M-H2O-H]-", "[M+ACN+H]+"]
_test_masses = np.array([50.0, 150.123, 499.9999, 1234.5678, 2000.0])
_all_adducts = list(dict.fromkeys(REF_ADDUCTS + ADDUCT_TABLE.loc[ADDUCT_TABLE.supported, "adduct"].tolist()))
rt_rows = []
for a in _all_adducts:
    p = parse_adduct(a)
    if not p["supported"]:
        continue
    err = np.max(np.abs(neutral_mass_from_precursor(precursor_from_neutral_mass(_test_masses, p), p) - _test_masses))
    rt_rows.append({"adduct": a, "n": p["n"], "z": p["z"], "shift": round(p["mass_shift"], 6), "max_roundtrip_err_da": err, "pass": err < 1e-9})
roundtrip = pd.DataFrame(rt_rows)
anchors = {
    "[M+H]+ @100": (precursor_from_neutral_mass(100.0, "[M+H]+"), 100 + PROTON_MASS),
    "[M-H]- @100": (precursor_from_neutral_mass(100.0, "[M-H]-"), 100 - PROTON_MASS),
    "[M+Na]+ @100": (precursor_from_neutral_mass(100.0, "[M+Na]+"), 100 + 22.9897692809 - ELECTRON_MASS),
    "[M+2H]2+ @100": (precursor_from_neutral_mass(100.0, "[M+2H]2+"), (100 + 2 * PROTON_MASS) / 2),
    "[2M+H]+ @100": (precursor_from_neutral_mass(100.0, "[2M+H]+"), 200 + PROTON_MASS),
    "[M+Cl]- @100": (precursor_from_neutral_mass(100.0, "[M+Cl]-"), 100 + 34.96885268 + ELECTRON_MASS),
}
anchor_ok = {k: abs(a - b) < 1e-7 for k, (a, b) in anchors.items()}
neg_cases = {"'[M+Xyz]+' unsupported": not parse_adduct("[M+Xyz]+")["supported"],
             "'M+H2O+Na' (no charge) unsupported": not parse_adduct("M+H2O+Na")["supported"],
             "'M+H' charge inferred +1": parse_adduct("M+H")["z"] == 1}
print(roundtrip.to_string(index=False))
for k, v in {**anchor_ok, **neg_cases}.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
check("Adduct algebra validated", roundtrip["pass"].all() and all(anchor_ok.values()) and all(neg_cases.values()),
      f"{len(roundtrip)} adducts round-trip < 1e-9 Da")

## 14. Validate adduct physics against real data

**QUESTION.** For labelled training spectra, does `neutral mass reconstructed from (precursor, adduct)` agree with the RDKit exact mass of the labelled structure?

Errors are reported in two spaces: **neutral** (`mass_error_da`, `mass_error_ppm` relative to M — the space in which candidates are searched) and **precursor** (`precursor_ppm` relative to the theoretical m/z — the instrument's native space). If the dataset ships its own ppm error, we test which of our definitions and sign conventions it matches.

In [ ]:
tm = train_meta
tm["mass_error_da"] = tm["neutral_mass_obs"] - tm["exact_mass"]
tm["mass_error_ppm"] = 1e6 * tm["mass_error_da"] / tm["exact_mass"]
tm["abs_mass_error_ppm"] = tm["mass_error_ppm"].abs()
if "adduct" in tm:
    aa = adduct_arrays(tm["adduct"])
    theo = (aa.n.values * tm.exact_mass.values + aa.mass_shift.values - aa.z.values * ELECTRON_MASS) / np.abs(aa.z.values)
    tm["precursor_ppm"] = 1e6 * (tm.precursor_mz.values - theo) / theo
else:
    tm["precursor_ppm"] = np.nan
_ok = tm.abs_mass_error_ppm.notna()
phys = {"spectra with reconstructable neutral mass": f"{_ok.mean():.2%}",
        "|neutral ppm| p50 / p95 / p99": " / ".join(f"{q:.2f}" for q in np.nanquantile(tm.abs_mass_error_ppm, [.5, .95, .99])) if _ok.any() else "n/a",
        "gross errors |ppm| > 50 (likely wrong adduct/label)": int((tm.abs_mass_error_ppm > 50).sum())}
if "ppm_provided" in tm and tm.ppm_provided.notna().any():
    best = None
    for name, ours in (("precursor_ppm", tm.precursor_ppm), ("mass_error_ppm", tm.mass_error_ppm)):
        for sgn in (1, -1):
            d = (sgn * ours - tm.ppm_provided).abs()
            med = float(np.nanmedian(d))
            if best is None or med < best[2]:
                best = (name, sgn, med, float(np.nanquantile(d, 0.95)), float(pd.Series(sgn * ours).corr(tm.ppm_provided, method="spearman")))
    phys.update({"provided ppm matches": f"{'' if best[1] == 1 else '−'}{best[0]}",
                 "median |disagreement| (ppm)": best[2], "p95 |disagreement| (ppm)": best[3], "Spearman r (robust to gross outliers)": best[4]})
    PHYSICS_AGREES = best[2] < 0.5 and best[4] > 0.95
    # Per-adduct disagreement localises any convention mismatch (e.g. an electron mass counted twice for one adduct).
    _d = (best[1] * tm[best[0]] - tm.ppm_provided)
    PHYS_BY_ADDUCT = (pd.DataFrame({"adduct": tm["adduct"].astype(str), "d": _d, "ad": _d.abs()}).dropna()
                      .groupby("adduct", observed=True)
                      .agg(n=("d", "size"), median_signed_disagreement_ppm=("d", "median"), p95_abs_disagreement_ppm=("ad", lambda s: s.quantile(.95)))
                      .sort_values("p95_abs_disagreement_ppm", ascending=False))
    PHYS_BY_ADDUCT["consistent"] = PHYS_BY_ADDUCT.median_signed_disagreement_ppm.abs() < 0.5
    display(PHYS_BY_ADDUCT.head(15))  # noqa: F821
    phys["adducts with systematic disagreement (|median| ≥ 0.5 ppm)"] = ", ".join(PHYS_BY_ADDUCT.index[~PHYS_BY_ADDUCT.consistent]) or "none"
else:
    PHYSICS_AGREES = bool(_ok.any() and np.nanquantile(tm.abs_mass_error_ppm, 0.5) < 5)
    phys["provided ppm"] = "not available; agreement judged from median |ppm| < 5"
report("§14 adduct physics vs data", phys,
       "PASS means our electron-mass-aware parser reproduces the dataset's chemistry conventions; the parser is the "
       "single source of truth for neutral-mass conversion in all later notebooks.")
CHECKS["Adduct physics matches data"] = bool(PHYSICS_AGREES)

## 15. Mass accuracy analysis and candidate tolerance

**QUESTION.** How accurate is the precursor mass, and which tolerance captures the true neutral mass for a target fraction of spectra?

Absolute errors are heavy-tailed, so percentiles (not means) are the primary summary. The tolerance is *measured*, not chosen: the recommended value is the smallest grid value capturing ≥ `target_capture` of training spectra in **neutral-mass ppm** (the space the candidate search uses).

In [ ]:
def ppm_summary(df, by, min_n=30, top=12):
    g = df.dropna(subset=["abs_mass_error_ppm"]).groupby(by, observed=True)["abs_mass_error_ppm"]
    out = g.quantile([.5, .9, .95, .99]).unstack()
    out.columns = ["p50", "p90", "p95", "p99"]
    out["n"] = g.size()
    return out[out.n >= min_n].sort_values("n", ascending=False).head(top).round(2)


err = tm.loc[tm.abs_mass_error_ppm.notna()]
if len(err):
    print("Global |neutral ppm|:", {k: round(v, 3) for k, v in qsummary(err.abs_mass_error_ppm, (.5, .9, .95, .99)).items()})
    print("Global |precursor ppm|:", {k: round(v, 3) for k, v in qsummary(err.precursor_ppm.abs(), (.5, .9, .95, .99)).items()})
    err = err.assign(mass_bin=pd.cut(err.exact_mass, [0, 200, 300, 400, 500, 700, 1000, np.inf]))
    for by in [c for c in ("adduct", "mass_bin", "ion_mode", "instrument", "source") if c in err]:
        print(f"\n|neutral ppm| by {by}")
        display(ppm_summary(err, by))  # noqa: F821
    cap = pd.DataFrame({"tol_ppm": CONFIG["tolerance_grid_ppm"]})
    cap["captured_neutral"] = [float((err.abs_mass_error_ppm <= t).mean()) for t in cap.tol_ppm]
    cap["captured_precursor"] = [float((err.precursor_ppm.abs() <= t).mean()) for t in cap.tol_ppm]
    if "source" in err:
        for s in err.source.value_counts().head(4).index:
            cap[f"captured[{s}]"] = [float((err.loc[err.source == s, "abs_mass_error_ppm"] <= t).mean()) for t in cap.tol_ppm]
    display(cap.round(4))  # noqa: F821
    _hit = cap[cap.captured_neutral >= CONFIG["target_capture"]]
    RECOMMENDED_PPM = float(_hit.tol_ppm.iloc[0]) if len(_hit) else float(cap.tol_ppm.iloc[-1])
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
    x = err.mass_error_ppm.clip(-50, 50)
    axes[0].hist(x, bins=200, color="#4472c4"); axes[0].set_title("Signed neutral-mass error (ppm, clipped ±50 for display)")
    axes[0].set_yscale("log")
    axes[1].plot(cap.tol_ppm, cap.captured_neutral, "o-", label="neutral ppm")
    axes[1].plot(cap.tol_ppm, cap.captured_precursor, "s--", label="precursor ppm")
    axes[1].axhline(CONFIG["target_capture"], color="grey", ls=":")
    axes[1].set_xscale("log"); axes[1].set_xlabel("tolerance (ppm)"); axes[1].set_ylabel("fraction captured"); axes[1].legend()
    axes[1].set_title(f"Capture vs tolerance → recommended ±{RECOMMENDED_PPM:g} ppm")
    savefig(fig, "15_mass_accuracy")
else:
    RECOMMENDED_PPM = 10.0
    cap = pd.DataFrame()

# neutral-mass agreement between spectra of the same unit (train: connectivity, test: prediction unit)
def within_group_spread_ppm(df, key):
    g = df.dropna(subset=["neutral_mass_obs"]).groupby(key, observed=True)["neutral_mass_obs"]
    st = g.agg(["min", "max", "median", "size"])
    st = st[st["size"] > 1]
    return 1e6 * (st["max"] - st["min"]) / st["median"]


spread_train = within_group_spread_ppm(tm.dropna(subset=["connectivity_key"]), "connectivity_key")
spread_test = within_group_spread_ppm(test_meta, "unit_id")
report("§15 mass accuracy", {
    "recommended candidate tolerance (neutral ppm)": RECOMMENDED_PPM,
    "capture at recommended": float(cap.loc[cap.tol_ppm == RECOMMENDED_PPM, "captured_neutral"].iloc[0]) if len(cap) else np.nan,
    "train: within-molecule neutral-mass spread p50/p95 (ppm)": " / ".join(f"{v:.2f}" for v in np.nanquantile(spread_train, [.5, .95])) if len(spread_train) else "n/a",
    "test: within-unit neutral-mass spread p50/p95 (ppm)": " / ".join(f"{v:.2f}" for v in np.nanquantile(spread_test, [.5, .95])) if len(spread_test) else "n/a (single-spectrum units)",
    "test units with spread > 2×tolerance": int((spread_test > 2 * RECOMMENDED_PPM).sum()) if len(spread_test) else 0,
}, f"Candidate search window = ±{RECOMMENDED_PPM:g} ppm in neutral-mass space (measured, not assumed). "
   "For multi-spectrum test units, a consensus (median) neutral mass is used; units whose spectra disagree beyond 2× tolerance are flagged.")
CHECKS["Mass-error analysis complete"] = bool(len(cap))

## 16. Collision-energy analysis

**QUESTION.** How is collision energy encoded (scalar, ramp, list), and in which units (eV, NCE %, V, unitless)?

**WHY IT MATTERS.** eV and normalised collision energy are not comparable, so they are never pooled into one histogram. Molecule-level summaries use the pair `(value, unit)`.

In [ ]:
def parse_collision_energy(x):
    out = {"ce_value": np.nan, "ce_min": np.nan, "ce_max": np.nan, "ce_unit": "missing", "ce_multi": False}
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return out
    s = str(x).strip().lower()
    if not s or s in ("nan", "none", "null"):
        return out
    nums = [float(v) for v in re.findall(r"\d+\.?\d*", s)]
    if "nce" in s or "%" in s or "normalized" in s or "normalised" in s:
        unit = "NCE"
    elif "ev" in s:
        unit = "eV"
    elif re.search(r"\d\s*v\b|\bv\b", s):
        unit = "V"
    elif nums and re.fullmatch(r"[\[\(]?\s*[\d.,\s\-]+[\]\)]?", s):
        unit = "unitless"
    else:
        unit = "unparsed:" + s[:20]
    if not nums:
        out["ce_unit"] = "unparsed:" + s[:20]
        return out
    out.update(ce_value=float(np.mean(nums)), ce_min=min(nums), ce_max=max(nums), ce_unit=unit, ce_multi=len(nums) > 1)
    return out


for d in (train_meta, test_meta):
    if "ce_raw" in d:
        u = d["ce_raw"].astype(object)
        tbl = pd.DataFrame([parse_collision_energy(v) for v in pd.unique(u)], index=pd.Index(pd.unique(u), dtype=object))
        parsed = tbl.reindex(u.values).reset_index(drop=True)
        for c in parsed.columns:
            d[c] = parsed[c].values
        d["ce_unit"] = d["ce_unit"].astype("category")
        d["ce_key"] = (d["ce_unit"].astype(str) + ":" + pd.Series(d["ce_value"]).round(1).astype(str).values).astype("category")
if "ce_raw" in train_meta:
    ce_units = pd.DataFrame({s: d.ce_unit.value_counts(normalize=True) for s, d in (("train", train_meta), ("test", test_meta)) if "ce_unit" in d}).fillna(0)
    display(ce_units.round(4))  # noqa: F821
    units = [u for u in ce_units.index if not u.startswith(("missing", "unparsed"))][:3]
    if units:
        fig, axes = plt.subplots(1, len(units), figsize=(5 * len(units), 3.2), squeeze=False)
        for ax, u in zip(axes[0], units):
            for s, d, col in (("train", train_meta, "#4472c4"), ("test", test_meta, "#ed7d31")):
                if "ce_unit" in d:
                    xv = d.loc[d.ce_unit == u, "ce_value"].dropna()
                    if len(xv):
                        ax.hist(xv, bins=60, alpha=0.6, density=True, color=col, label=f"{s} (n={len(xv)})")
            ax.set_title(f"collision energy, unit = {u}"); ax.legend(fontsize=8)
        savefig(fig, "16_collision_energy_by_unit")
    ce_per_mol = train_meta.dropna(subset=["connectivity_key"]).groupby("connectivity_key", observed=True)["ce_key"].nunique()
    report("§16 collision energy", {"units (train share)": ce_units["train"].round(4).to_dict(),
                                    "multi-energy / ramp spectra (train)": f"{train_meta.ce_multi.mean():.2%}",
                                    "distinct (value,unit) per molecule p50/p90": f"{ce_per_mol.median():.0f} / {ce_per_mol.quantile(.9):.0f}"},
           "Collision energy is represented as (value, unit, is_multi); units are never pooled. "
           "No cross-unit normalisation is applied here because no instrument-specific conversion is scientifically justified from these data alone.")
else:
    report("§16 collision energy", "No collision-energy column in the data.")

## 18. Peak-array sampling strategy *(executed before §17 because the audit needs peaks)*

**QUESTION.** How do we look at peaks without loading millions of arrays?

**ANALYSIS.** Peak counts for **all** spectra were already streamed from Arrow list lengths (§4). Peak *values* are loaded for a bounded, seeded sample with two components:

* **A — stratified spectra:** stratified by `split × source × precursor-m/z decile`, proportional allocation (largest-remainder), so the sample mirrors each split's mass and source mix.
* **B — molecule panel:** multi-spectrum training molecules with up to *k* spectra each, needed for same-molecule comparisons (§21–24). A purely random spectrum sample would contain almost no same-molecule pairs.

Test spectra are taken in full up to a cap. Sample metadata are saved.

In [ ]:
def proportional_quota(sizes: pd.Series, total):
    raw = sizes / sizes.sum() * total
    q = np.floor(raw).astype(int)
    rem = int(total - q.sum())
    if rem > 0:
        q[(raw - q).sort_values(ascending=False).index[:rem]] += 1
    return np.minimum(q, sizes)


def stratified_sample(df, total, strata_cols, seed):
    if total >= len(df):
        return df.index.to_numpy()
    rng = np.random.default_rng(seed)
    key = df[strata_cols].astype(str).agg("|".join, axis=1)
    quota = proportional_quota(key.value_counts(), total)
    r = pd.Series(rng.random(len(df)), index=df.index)
    order = r.groupby(key).rank(method="first")
    return df.index[(order <= key.map(quota)).to_numpy()].to_numpy()


_edges = np.nanquantile(pd.concat([train_meta.precursor_mz, test_meta.precursor_mz]), np.linspace(0, 1, 11))
for d in (train_meta, test_meta):
    d["mz_decile"] = np.clip(np.searchsorted(_edges[1:-1], d.precursor_mz.fillna(np.nanmedian(d.precursor_mz)).values, side="right"), 0, 9)
_strata = ["mz_decile"] + (["source"] if "source" in train_meta else [])

idx_A = stratified_sample(train_meta, CONFIG["peak_sample_train_spectra"], _strata, SEED)
multi = per_mol.index[per_mol.n_spectra >= 2]
_rng = np.random.default_rng(SEED + 1)
panel = _rng.choice(np.asarray(multi, dtype=object), size=min(len(multi), CONFIG["peak_sample_train_molecules"]), replace=False) if len(multi) else []
_pm = train_meta[train_meta.connectivity_key.isin(panel)]
_pm = _pm.assign(_r=np.random.default_rng(SEED + 2).random(len(_pm))).sort_values("_r")
idx_B = _pm.groupby("connectivity_key", observed=True).head(CONFIG["peak_sample_spectra_per_molecule"]).index.to_numpy()
idx_T = stratified_sample(test_meta, CONFIG["peak_sample_test_max"], ["mz_decile"], SEED)

train_s = train_meta.loc[np.union1d(idx_A, idx_B)].copy()
train_s["sample_component"] = np.where(train_s.index.isin(idx_B), np.where(train_s.index.isin(idx_A), "A+B", "B"), "A")
test_s = test_meta.loc[np.sort(idx_T)].copy()
test_s["sample_component"] = "T"
print(f"sampled train spectra: {len(train_s):,} (A={len(idx_A):,}, B={len(idx_B):,}); test: {len(test_s):,}")


def read_peaks(split, positions=None, ids=None):
    """Load (mz, intensity) arrays only for the requested rows, streaming parquet batches."""
    src = PEAK_SRC[split]
    cols = [c for c in (src["mz"], src["int"], src["peaks"]) if c]
    if src["id_col"]:
        id_all = load_table(src["path"], columns=[src["id_col"]])[src["id_col"]].astype(str)
        positions = pd.Series(np.arange(len(id_all)), index=id_all.values).reindex(pd.Index(ids).astype(str)).to_numpy()
    positions = np.asarray(positions, dtype=np.int64)
    order = np.argsort(positions)
    sorted_pos = positions[order]
    out = [None] * len(positions)

    def _emit(k, rowvals):
        mzc = rowvals.get(src["mz"]) if src["mz"] else None
        itc = rowvals.get(src["int"]) if src["int"] else None
        pkc = rowvals.get(src["peaks"]) if src["peaks"] else None
        out[order[k]] = split_peaks(mzc, itc, pkc)

    if _is_parquet_path(src["path"]):
        dset = pads.dataset(str(src["path"]), format="parquet")
        offset = 0
        for b in dset.to_batches(columns=cols, batch_size=65_536):
            lo_, hi_ = np.searchsorted(sorted_pos, [offset, offset + b.num_rows])
            if hi_ > lo_:
                local = sorted_pos[lo_:hi_] - offset
                sub = b.take(pa.array(local))
                colvals = {c: sub.column(c).to_pylist() for c in cols}
                for j, k in enumerate(range(lo_, hi_)):
                    _emit(k, {c: colvals[c][j] for c in cols})
            offset += b.num_rows
    else:
        full = load_table(src["path"], columns=cols)
        for k, p in enumerate(sorted_pos):
            _emit(k, {c: full[c].iat[p] for c in cols})
    return out


t0 = time.time()
PEAKS_TRAIN = read_peaks("train", positions=train_s.row_pos.values, ids=train_s.spectrum_id.values)
PEAKS_TEST = read_peaks("test", positions=test_s.row_pos.values, ids=test_s.spectrum_id.values)
assert all(p is not None for p in PEAKS_TRAIN + PEAKS_TEST), "some sampled rows were not found in the peak source"
print(f"peaks loaded in {time.time()-t0:.1f}s")

sample_meta = pd.concat([train_s[["spectrum_id", "split", "sample_component", "mz_decile"] + (["source"] if "source" in train_s else [])],
                         test_s[["spectrum_id", "split", "sample_component", "mz_decile"]]], ignore_index=True)
SAMPLE_INFO = {"SEED": SEED, "method": "A: stratified (split×source×m/z decile, proportional largest-remainder); "
               "B: seeded multi-spectrum molecule panel, ≤k spectra each; T: test stratified by m/z decile (full if ≤ cap)",
               "n_train": len(train_s), "n_test": len(test_s),
               "source_proportions_sample": (train_s.source.value_counts(normalize=True).round(4).to_dict() if "source" in train_s else None),
               "source_proportions_population": (train_meta.source.value_counts(normalize=True).round(4).to_dict() if "source" in train_meta else None)}
save_artifact(sample_meta, "peak_sample_metadata", extra={"sampling": SAMPLE_INFO})
report("§18 sampling", {k: v for k, v in SAMPLE_INFO.items() if k != "method"}, "Sample definition saved to peak_sample_metadata.parquet (+ .meta.json).")

## 17. Spectral data quality

**QUESTION.** Are peak arrays well-formed, and how often do physically odd spectra occur?

Anomalies are **flagged and quantified, never deleted**. Global counts (empty / 1-peak) use the full-population peak counts from §4; everything else uses the §18 sample.

In [ ]:
def compute_spectrum_features(mz, inten, precursor=np.nan):
    f = {"len_mismatch": len(mz) != len(inten)}
    m = min(len(mz), len(inten))
    mz, it = np.asarray(mz[:m], float), np.asarray(inten[:m], float)
    finite = np.isfinite(mz) & np.isfinite(it)
    f["n_nan"] = int((~finite).sum())
    f["n_zero_int"] = int((it[finite] == 0).sum())
    f["n_neg_int"] = int((it[finite] < 0).sum())
    f["unsorted_mz"] = bool(m > 1 and np.any(np.diff(mz[finite]) < 0))
    f["n_dup_mz"] = int(finite.sum() - len(np.unique(mz[finite])))
    ok = finite & (it > 0)
    mz, it = mz[ok], it[ok]
    n = len(mz)
    f["n_peaks_raw"] = m
    f["n_peaks"] = n
    if n == 0:
        return {**f, **dict.fromkeys(["mz_min", "mz_max", "mz_mean", "mz_median", "mz_std", "base_peak_mz", "base_peak_intensity",
                                      "top1_intensity_fraction", "top3_intensity_fraction", "top5_intensity_fraction",
                                      "top10_intensity_fraction", "spectral_entropy", "normalized_entropy",
                                      "fragment_mass_range", "max_fragment_over_precursor", "fragment_coverage"], np.nan),
                "n_above_precursor": 0, "n_extreme_mz": 0}
    s = it.sum()
    p = it / s
    srt = np.sort(p)[::-1]
    cs = np.cumsum(srt)
    ent = float(-(p * np.log(p)).sum())
    bi = int(np.argmax(it))
    f.update(mz_min=mz.min(), mz_max=mz.max(), mz_mean=mz.mean(), mz_median=float(np.median(mz)), mz_std=mz.std(),
             base_peak_mz=mz[bi], base_peak_intensity=it[bi],
             top1_intensity_fraction=cs[0], top3_intensity_fraction=cs[min(2, n - 1)],
             top5_intensity_fraction=cs[min(4, n - 1)], top10_intensity_fraction=cs[min(9, n - 1)],
             spectral_entropy=ent, normalized_entropy=ent / np.log(n) if n > 1 else 0.0,
             fragment_mass_range=mz.max() - mz.min(),
             max_fragment_over_precursor=mz.max() / precursor if precursor and np.isfinite(precursor) and precursor > 0 else np.nan,
             fragment_coverage=(mz.max() - mz.min()) / precursor if precursor and np.isfinite(precursor) and precursor > 0 else np.nan,
             n_above_precursor=int((mz > precursor + CONFIG["fragment_above_precursor_da"]).sum()) if np.isfinite(precursor) else 0,
             n_extreme_mz=int((mz > CONFIG["extreme_fragment_mz"]).sum()))
    return f


def features_for(meta_s, peaks):
    rows = [compute_spectrum_features(mz, it, pr) for (mz, it), pr in zip(peaks, meta_s.precursor_mz.values)]
    return pd.DataFrame(rows, index=meta_s.index)


t0 = time.time()
feat_train = features_for(train_s, PEAKS_TRAIN)
feat_test = features_for(test_s, PEAKS_TEST)
print(f"features computed in {time.time()-t0:.1f}s")
FEAT_COLS = list(feat_train.columns)
train_s = train_s.join(feat_train)
test_s = test_s.join(feat_test)


def audit(df, split, pop):
    n = len(df)
    npk = pop.n_peaks_all
    has_pop = npk.notna().any()
    return {
        "split": split, "n_sampled": n,
        "empty spectra (population)": int((npk == 0).sum()) if has_pop else int((df.n_peaks_raw == 0).sum()),
        "1-peak spectra (population)": int((npk == 1).sum()) if has_pop else int((df.n_peaks_raw == 1).sum()),
        "zero-intensity peaks (spectra %)": f"{(df.n_zero_int > 0).mean():.2%}",
        "negative intensities (spectra %)": f"{(df.n_neg_int > 0).mean():.2%}",
        "NaN peaks (spectra %)": f"{(df.n_nan > 0).mean():.2%}",
        "duplicate m/z (spectra %)": f"{(df.n_dup_mz > 0).mean():.2%}",
        "unsorted m/z (spectra %)": f"{df.unsorted_mz.mean():.2%}",
        "m/z–intensity length mismatch": int(df.len_mismatch.sum()),
        f"fragments > precursor+{CONFIG['fragment_above_precursor_da']:g} Da (spectra %)": f"{(df.n_above_precursor > 0).mean():.2%}",
        f"fragments > {CONFIG['extreme_fragment_mz']:g} m/z (spectra)": int((df.n_extreme_mz > 0).sum()),
    }


audit_tbl = pd.DataFrame([audit(train_s, "train", train_meta), audit(test_s, "test", test_meta)]).set_index("split").T
display(audit_tbl)  # noqa: F821
_severe = int(train_s.len_mismatch.sum() + test_s.len_mismatch.sum())
report("§17 spectral quality", {"n_peaks train (population) p50/p95": (f"{train_meta.n_peaks_all.median():.0f} / {train_meta.n_peaks_all.quantile(.95):.0f}"
                                                                        if train_meta.n_peaks_all.notna().any() else "sample only"),
                                "n_peaks test (population) p50/p95": (f"{test_meta.n_peaks_all.median():.0f} / {test_meta.n_peaks_all.quantile(.95):.0f}"
                                                                       if test_meta.n_peaks_all.notna().any() else "sample only"),
                                "length mismatches (sample)": _severe},
       "Canonical peak cleaning for all later notebooks: drop non-finite and non-positive intensities, sort by m/z, "
       "merge exact-duplicate m/z by summing. Fragments above the precursor are kept but flagged (they may be isotopes/contaminants).")
CHECKS["Spectral quality audited"] = True

## 19. Spectral complexity

**QUESTIONS.** Do larger molecules produce more peaks? Do some acquisition settings give sparse spectra? Are some adducts systematically more complex? Does test look more complex than train?

In [ ]:
both_s = pd.concat([train_s, test_s])
rho_tr = stats.spearmanr(train_s.precursor_mz, train_s.n_peaks, nan_policy="omit").statistic
rho_te = stats.spearmanr(test_s.precursor_mz, test_s.n_peaks, nan_policy="omit").statistic
CPLX = ["n_peaks", "spectral_entropy", "top1_intensity_fraction", "fragment_coverage"]
print(f"Spearman(precursor m/z, n_peaks): train {rho_tr:.3f} | test {rho_te:.3f}")
print("\nTrain vs test medians:")
display(both_s.groupby("split")[CPLX].median().round(3))  # noqa: F821
for by in [c for c in ("adduct", "ion_mode", "ce_unit", "instrument", "source") if c in both_s]:
    g = both_s.groupby([by, "split"], observed=True)[CPLX].median().round(3)
    g["n"] = both_s.groupby([by, "split"], observed=True).size()
    g = g[g.n >= 30].sort_values("n", ascending=False).head(12)
    print(f"\nComplexity by {by}")
    display(g)  # noqa: F821
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].hexbin(train_s.precursor_mz, np.log10(train_s.n_peaks.clip(lower=1)), gridsize=50, bins="log")
axes[0].set_xlabel("precursor m/z"); axes[0].set_ylabel("log10 n_peaks"); axes[0].set_title(f"train: n_peaks vs mass (ρ={rho_tr:.2f})")
for s, d in (("train", train_s), ("test", test_s)):
    axes[1].hist(np.log10(d.n_peaks.clip(lower=1)), bins=60, alpha=0.6, density=True, label=s)
    axes[2].hist(d.spectral_entropy.dropna(), bins=60, alpha=0.6, density=True, label=s)
axes[1].set_title("log10 n_peaks"); axes[1].legend(); axes[2].set_title("spectral entropy (nats)"); axes[2].legend()
savefig(fig, "19_spectral_complexity")
if "adduct" in train_s:
    top_add = train_s.adduct.value_counts().head(8).index
    fig, ax = plt.subplots(figsize=(13, 3.2))
    data = [np.log10(train_s.loc[train_s.adduct == a, "n_peaks"].clip(lower=1)) for a in top_add]
    ax.boxplot(data, showfliers=False); ax.set_xticks(range(1, len(data) + 1), [str(a) for a in top_add]); ax.set_ylabel("log10 n_peaks"); ax.set_title("Peak count by adduct (train sample)")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    savefig(fig, "19_npeaks_by_adduct")
report("§19 complexity", {"Spearman mass–n_peaks (train)": float(rho_tr), "Spearman mass–n_peaks (test)": float(rho_te),
                          "median n_peaks train/test": f"{train_s.n_peaks.median():.0f} / {test_s.n_peaks.median():.0f}",
                          "median entropy train/test": f"{train_s.spectral_entropy.median():.2f} / {test_s.spectral_entropy.median():.2f}"},
       "Peak-count and entropy differences by acquisition are recorded as covariates; they must be model inputs (or normalised away), "
       "not ignored, because they change what 'a match' looks like.")

## 20. Representative spectra

A single reusable `plot_spectrum()` is used for every spectrum figure in the project. Each panel prints its metadata.

In [ ]:
def plot_spectrum(ax, mz, inten, title="", precursor=None, meta=None, annotate_top=5, normalize=True, color="#1f4e79", invert=False):
    mz, it = clean_peaks(mz, inten)
    if len(mz) == 0:
        ax.text(0.5, 0.5, "empty spectrum", ha="center", transform=ax.transAxes)
        return ax
    y = it / it.max() * 100 if normalize else it
    y = -y if invert else y
    ax.vlines(mz, 0, y, color=color, lw=0.8)
    if precursor is not None and np.isfinite(precursor):
        ax.axvline(precursor, color="red", ls="--", lw=0.8)
    for i in np.argsort(it)[::-1][:annotate_top]:
        ax.text(mz[i], y[i], f"{mz[i]:.3f}", fontsize=6, ha="center", va="bottom" if not invert else "top")
    ax.set_title(title, fontsize=8)
    if meta:
        ax.text(0.99, 0.97, "\n".join(f"{k}: {v}" for k, v in meta.items()), transform=ax.transAxes, fontsize=6,
                ha="right", va="top", bbox=dict(fc="white", ec="0.8", alpha=0.8))
    ax.set_xlabel("m/z", fontsize=7); ax.tick_params(labelsize=6)
    return ax


def clean_peaks(mz, it, max_peaks=None):
    mz, it = np.asarray(mz, float), np.asarray(it, float)
    m = min(len(mz), len(it))
    mz, it = mz[:m], it[:m]
    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0) & (mz > 0)
    mz, it = mz[ok], it[ok]
    if len(mz) == 0:
        return mz, it
    o = np.argsort(mz, kind="stable")
    mz, it = mz[o], it[o]
    if len(mz) > 1 and np.any(np.diff(mz) == 0):
        mz, inv = np.unique(mz, return_inverse=True)
        it = np.bincount(inv, weights=it)
    if max_peaks and len(mz) > max_peaks:
        keep = np.sort(np.argsort(it)[-max_peaks:])
        mz, it = mz[keep], it[keep]
    return mz, it


def _meta_of(r):
    keys = [k for k in ("adduct", "ion_mode", "ce_raw", "instrument", "source") if k in r.index]
    m = {k: r[k] for k in keys}
    m.update(n_peaks=int(r.n_peaks), entropy=f"{r.spectral_entropy:.2f}" if np.isfinite(r.spectral_entropy) else "nan")
    return m


_pos = train_s.reset_index(drop=True)
_sel = {}
_r = np.random.default_rng(SEED)
def _pick(mask):
    ii = np.flatnonzero(mask.to_numpy() & (_pos.n_peaks > 0).to_numpy())
    return int(_r.choice(ii)) if len(ii) else None

_sel["low-mass"] = _pick(_pos.precursor_mz <= _pos.precursor_mz.quantile(.05))
_sel["high-mass"] = _pick(_pos.precursor_mz >= _pos.precursor_mz.quantile(.95))
_sel["few peaks"] = _pick(_pos.n_peaks.between(2, 4))
_sel["many peaks"] = _pick(_pos.n_peaks >= _pos.n_peaks.quantile(.99))
if "ion_mode" in _pos:
    _sel["positive mode"] = _pick(_pos.ion_mode == "positive")
    _sel["negative mode"] = _pick(_pos.ion_mode == "negative")
if "adduct" in _pos:
    vc = _pos.adduct.value_counts()
    vc = vc[vc > 0]
    _sel["common adduct"] = _pick(_pos.adduct == vc.index[0])
    _sel["rare adduct"] = _pick(_pos.adduct == vc.index[-1])
_sel["low entropy"] = _pick(_pos.spectral_entropy <= _pos.spectral_entropy.quantile(.05))
_sel["high entropy"] = _pick(_pos.spectral_entropy >= _pos.spectral_entropy.quantile(.95))
_sel = {k: v for k, v in _sel.items() if v is not None}
nrow = math.ceil(len(_sel) / 2)
fig, axes = plt.subplots(nrow, 2, figsize=(14, 2.6 * nrow), squeeze=False)
for ax, (label, i) in zip(axes.ravel(), _sel.items()):
    r = _pos.iloc[i]
    plot_spectrum(ax, *PEAKS_TRAIN[i], title=f"{label} — precursor {r.precursor_mz:.4f}", precursor=r.precursor_mz, meta=_meta_of(r))
for ax in axes.ravel()[len(_sel):]:
    ax.axis("off")
fig.tight_layout()
savefig(fig, "20_representative_spectra")

### Similarity toolkit and controlled pair dataset (used by §21–24)

* `binned_matrix()` → sparse CSR, bins up to a controlled `MAX_FRAGMENT_MZ` (percentile-based; the fraction of peaks beyond it is reported), never sized by one extreme m/z.
* `cosine_greedy()` / modified cosine: peak-aware matching within ±`peak_match_tol_da`, greedy one-to-one assignment (the matchms algorithm), cross-checked against matchms when installed.
* `entropy_similarity()`: unweighted spectral-entropy similarity (Li et al., 2021) on sum-normalised spectra.

**Pairs.** Positives = same connectivity, different spectra. Negatives get progressively harder, all anchored on the *same* anchor spectra as the positives so that only the partner changes:

| set | partner constraint (different connectivity) |
|---|---|
| N1 | random |
| N2 | precursor m/z within ±tol ppm |
| N3 | neutral (exact) mass within ±tol ppm |
| N4 | N3 + same ion mode |
| N5 | N4 + same adduct |
| N6 | N5 + same source, instrument and collision energy |

In [ ]:
PEAKS_TRAIN_CLEAN = [clean_peaks(mz, it, CONFIG["max_peaks_similarity"]) for mz, it in PEAKS_TRAIN]
_allfrag = np.concatenate([p[0] for p in PEAKS_TRAIN_CLEAN if len(p[0])]) if PEAKS_TRAIN_CLEAN else np.array([0.0])
MAX_FRAGMENT_MZ = float(min(3000.0, 50 * math.ceil(np.quantile(_allfrag, 0.999) / 50)))
FRAC_PEAKS_BEYOND = float((_allfrag > MAX_FRAGMENT_MZ).mean())
print(f"MAX_FRAGMENT_MZ = {MAX_FRAGMENT_MZ:g} (p99.9 of sampled fragments, rounded up to 50); peaks beyond: {FRAC_PEAKS_BEYOND:.4%}")

INTENSITY_TRANSFORMS = {
    "raw": lambda x: x,
    "max_norm": lambda x: x / x.max(),
    "sum_norm": lambda x: x / x.sum(),
    "sqrt": np.sqrt,
    "log1p_raw": np.log1p,
    "log1p_max100": lambda x: np.log1p(100 * x / x.max()),
    "sqrt_max_norm": lambda x: np.sqrt(x / x.max()),
}


def binned_matrix(peaks, width, max_mz=None, transform="sqrt_max_norm", norm="l2"):
    max_mz = max_mz or MAX_FRAGMENT_MZ
    f = INTENSITY_TRANSFORMS[transform]
    rows, cols, vals = [], [], []
    for i, (mz, it) in enumerate(peaks):
        keep = mz <= max_mz
        if not keep.any():
            continue
        rows.append(np.full(int(keep.sum()), i, dtype=np.int32))
        cols.append((mz[keep] / width).astype(np.int32))
        vals.append(f(it[keep]).astype(np.float64))
    ncol = int(max_mz / width) + 1
    if not rows:
        return sp.csr_matrix((len(peaks), ncol))
    X = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(len(peaks), ncol))
    X.sum_duplicates()
    if norm == "l2":
        nrm = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
    else:
        nrm = np.asarray(X.sum(axis=1)).ravel()
    nrm[nrm == 0] = 1
    return sp.csr_matrix(sp.diags(1 / nrm) @ X)


def pair_cosine(X, i, j):
    return np.asarray(X[i].multiply(X[j]).sum(axis=1)).ravel()


def _row_entropy(M):
    d = M.data.copy()
    v = np.zeros_like(d)
    nz = d > 0
    v[nz] = -d[nz] * np.log(d[nz])
    return np.asarray(sp.csr_matrix((v, M.indices, M.indptr), shape=M.shape).sum(axis=1)).ravel()


def entropy_similarity(P, i, j):
    Pi, Pj = P[i], P[j]
    return 1 - (2 * _row_entropy((Pi + Pj) * 0.5) - _row_entropy(Pi) - _row_entropy(Pj)) / np.log(4)


def _match_pairs(mz1, mz2, tol, shift=0.0):
    lo_ = np.searchsorted(mz2 + shift, mz1 - tol, "left")
    hi_ = np.searchsorted(mz2 + shift, mz1 + tol, "right")
    cnt = hi_ - lo_
    if cnt.sum() == 0:
        return np.empty(0, int), np.empty(0, int)
    a = np.repeat(np.arange(len(mz1)), cnt)
    b = np.arange(cnt.sum()) - np.repeat(np.cumsum(cnt) - cnt, cnt) + np.repeat(lo_, cnt)
    return a, b


def cosine_greedy(mz1, i1, mz2, i2, tol=None, precursor_shift=None):
    """Greedy peak-matching cosine; with precursor_shift=prec1-prec2 it becomes the modified cosine."""
    tol = tol or CONFIG["peak_match_tol_da"]
    if len(mz1) == 0 or len(mz2) == 0:
        return 0.0
    a, b = _match_pairs(mz1, mz2, tol)
    if precursor_shift is not None and abs(precursor_shift) > tol:
        a2, b2 = _match_pairs(mz1, mz2, tol, shift=precursor_shift)
        a, b = np.concatenate([a, a2]), np.concatenate([b, b2])
    if len(a) == 0:
        return 0.0
    sc = i1[a] * i2[b]
    o = np.lexsort((b, a, -sc))
    used1, used2, tot = set(), set(), 0.0
    for k in o:
        if a[k] in used1 or b[k] in used2:
            continue
        used1.add(a[k]); used2.add(b[k]); tot += sc[k]
    return float(tot / (np.sqrt((i1 ** 2).sum()) * np.sqrt((i2 ** 2).sum())))


def compute_spectral_similarity(peaks_a, peaks_b, method="cosine", transform="sqrt_max_norm", precursor_a=None, precursor_b=None):
    (m1, i1), (m2, i2) = peaks_a, peaks_b
    if len(m1) == 0 or len(m2) == 0:
        return 0.0
    f = INTENSITY_TRANSFORMS[transform]
    shift = (precursor_a - precursor_b) if method == "modified_cosine" else None
    return cosine_greedy(m1, f(i1), m2, f(i2), precursor_shift=shift)


# ---------------- pair dataset ----------------
S = train_s.reset_index(drop=True)
S_conn = S.connectivity_key.astype(object).to_numpy()
_valid = pd.notna(S_conn) & (S.n_peaks.to_numpy() > 0)


def _eq(col):
    if col not in S:
        return None
    return S[col].astype(object).to_numpy()


def build_pair_dataset(n_anchors, tol_ppm, seed):
    rng = np.random.default_rng(seed)
    idx = np.flatnonzero(_valid)
    grp = pd.Series(idx).groupby(S_conn[idx]).apply(list)
    pos = []
    for members in grp:
        if len(members) < 2:
            continue
        m = rng.permutation(members)
        for k in range(min(3, len(m) - 1)):
            pos.append((m[k], m[k + 1]))
    pos = np.array(pos, dtype=np.int64).reshape(-1, 2)
    if len(pos) > n_anchors:
        pos = pos[rng.choice(len(pos), n_anchors, replace=False)]
    em, pmz = S.exact_mass.to_numpy(), S.precursor_mz.to_numpy()
    o_em = idx[np.argsort(em[idx])]; s_em = em[o_em]
    o_pm = idx[np.argsort(pmz[idx])]; s_pm = pmz[o_pm]
    mode, add, src, ins, ce = _eq("ion_mode"), _eq("adduct"), _eq("source"), _eq("instrument"), _eq("ce_key")
    rows = [(a, b, "POS", 1) for a, b in pos]
    for a in pos[:, 0]:
        # N1 random
        for _ in range(20):
            b = int(rng.choice(idx))
            if S_conn[b] != S_conn[a]:
                rows.append((a, b, "N1 random", 0)); break
        for name, order, svals, center in (("N2 ~precursor", o_pm, s_pm, pmz[a]), ("N3 ~neutral", o_em, s_em, em[a])):
            lo_, hi_ = np.searchsorted(svals, [center * (1 - tol_ppm * 1e-6), center * (1 + tol_ppm * 1e-6)])
            cand = order[lo_:hi_]
            cand = cand[S_conn[cand] != S_conn[a]]
            if name == "N3 ~neutral":
                base = cand
                tiers = [("N3 ~neutral", np.ones(len(base), bool))]
                m4 = (mode[base] == mode[a]) if mode is not None else np.ones(len(base), bool)
                tiers.append(("N4 +ion mode", m4))
                m5 = m4 & ((add[base] == add[a]) if add is not None else True)
                tiers.append(("N5 +adduct", m5))
                m6 = m5.copy()
                for arr in (src, ins, ce):
                    if arr is not None:
                        m6 &= arr[base] == arr[a]
                tiers.append(("N6 +acquisition", m6))
                for tname, msk in tiers:
                    c = base[msk]
                    if len(c):
                        rows.append((a, int(rng.choice(c)), tname, 0))
            elif len(cand):
                rows.append((a, int(rng.choice(cand)), name, 0))
    P = pd.DataFrame(rows, columns=["i", "j", "set", "label"])
    for col, arr in (("same_adduct", add), ("same_ion_mode", mode), ("same_source", src), ("same_instrument", ins), ("same_ce", ce)):
        if arr is not None:
            P[col] = arr[P.i.values] == arr[P.j.values]
    return P


PAIRS = build_pair_dataset(CONFIG["n_pair_anchors"], CONFIG["pair_mass_tol_ppm"], SEED)
NEG_SETS = [s for s in ["N1 random", "N2 ~precursor", "N3 ~neutral", "N4 +ion mode", "N5 +adduct", "N6 +acquisition"] if s in set(PAIRS.set)]
print(PAIRS.set.value_counts().to_string())

# provisional representation (fixed BEFORE looking at results, refined in §22–23)
X_PROV = binned_matrix(PEAKS_TRAIN_CLEAN, CONFIG["provisional_bin_width"], transform="sqrt_max_norm")
PAIRS["cos_prov"] = pair_cosine(X_PROV, PAIRS.i.values, PAIRS.j.values)

## 21. Same molecule, different acquisition

**QUESTIONS.** How stable is the MS/MS fingerprint of one molecule? Which acquisition differences change it most?

Examples are chosen among molecules whose sampled spectra *actually differ* in adduct, collision energy or instrument (ranked by number of distinct acquisition signatures), not just any molecule with ≥ 3 spectra. The quantitative answer uses all positive pairs, grouped by what differs, scored with the provisional representation (0.1 Da bins, √max-normalised cosine).

In [ ]:
sig_cols = [c for c in ("adduct", "ce_key", "instrument") if c in S]
if sig_cols and len(S):
    sig = S.loc[_valid, sig_cols + ["connectivity_key"]].astype(str)
    div = sig.drop_duplicates().groupby("connectivity_key").size().sort_values(ascending=False)
    div = div[div >= 2]
    chosen = list(div.head(20).sample(min(4, len(div)), random_state=SEED).index) if len(div) else []
    if chosen:
        fig, axes = plt.subplots(len(chosen), 4, figsize=(16, 2.4 * len(chosen)), squeeze=False)
        for r_, ck in enumerate(chosen):
            ii = S.index[(S.connectivity_key.astype(str) == ck) & _valid]
            ii = sig.loc[ii].drop_duplicates(subset=sig_cols).index[:4]
            for c_ in range(4):
                ax = axes[r_, c_]
                if c_ < len(ii):
                    k = ii[c_]
                    rr = S.loc[k]
                    plot_spectrum(ax, *PEAKS_TRAIN[k], precursor=rr.precursor_mz,
                                  title=f"{ck} | cos to 1st = {pair_cosine(X_PROV, np.array([ii[0]]), np.array([k]))[0]:.2f}",
                                  meta={c: rr[c] for c in sig_cols})
                else:
                    ax.axis("off")
        fig.tight_layout()
        savefig(fig, "21_same_molecule_different_acquisition")
pos_pairs = PAIRS[PAIRS.label == 1].copy()


def _what_differs(r):
    diffs = [n for n, c in (("adduct", "same_adduct"), ("instrument", "same_instrument"), ("source", "same_source"), ("CE", "same_ce")) if c in r and not r[c]]
    if not diffs:
        return "identical acquisition"
    if diffs == ["CE"]:
        return "CE only"
    return diffs[0] + " differs" if len(diffs) == 1 else "+".join(diffs) + " differ"


if len(pos_pairs):
    pos_pairs["differs"] = pos_pairs.apply(_what_differs, axis=1)
    stab = pos_pairs.groupby("differs")["cos_prov"].agg(n="size", median="median", p25=lambda x: x.quantile(.25), p75=lambda x: x.quantile(.75)).sort_values("median", ascending=False)
    stab = stab[stab.n >= 10]
    display(stab.round(3))  # noqa: F821
    worst = stab["median"].idxmin() if len(stab) else None
    report("§21 same-molecule stability", {"positive pairs": len(pos_pairs),
                                           "median cosine, identical acquisition": float(stab["median"].get("identical acquisition", np.nan)),
                                           "median cosine, CE only differs": float(stab["median"].get("CE only", np.nan)),
                                           "median cosine, adduct differs": float(stab["median"].get("adduct differs", np.nan)),
                                           "largest change caused by": worst},
           "Acquisition context must be an input to any spectrum-to-structure model; comparisons across the condition "
           "that causes the largest change should not be trusted without conditioning on it.")

## 22. Spectral normalisation study

**QUESTION.** Which intensity transform separates same-molecule from different-molecule pairs best?

**Note on scale invariance.** Cosine similarity is invariant to multiplying a spectrum by a constant, so `raw`, `max_norm` and `sum_norm` are *mathematically identical* under cosine; only non-linear transforms (√, log) can change it. `log1p_raw` additionally depends on the absolute intensity scale (instrument-dependent). Entropy similarity is evaluated too because it is sensitive to the intensity shape. Choice = highest mean AUC over the hard negative sets (N3–N6).

In [ ]:
HARD_SETS = [s for s in NEG_SETS if s[:2] in ("N3", "N4", "N5", "N6")] or NEG_SETS


def auc_by_set(P, score):
    out = {}
    pos_s = P.loc[P.label == 1, score].to_numpy()
    for s in NEG_SETS:
        neg = P.loc[P.set == s, score].to_numpy()
        if len(pos_s) >= 10 and len(neg) >= 10:
            out[s] = roc_auc_score(np.r_[np.ones(len(pos_s)), np.zeros(len(neg))], np.r_[pos_s, neg])
    return out


norm_rows = []
for tname in INTENSITY_TRANSFORMS:
    Xc = binned_matrix(PEAKS_TRAIN_CLEAN, CONFIG["provisional_bin_width"], transform=tname, norm="l2")
    Xs = binned_matrix(PEAKS_TRAIN_CLEAN, CONFIG["provisional_bin_width"], transform=tname, norm="sum")
    PAIRS[f"cos[{tname}]"] = pair_cosine(Xc, PAIRS.i.values, PAIRS.j.values)
    PAIRS[f"ent[{tname}]"] = entropy_similarity(Xs, PAIRS.i.values, PAIRS.j.values)
    for metric in ("cos", "ent"):
        a = auc_by_set(PAIRS, f"{metric}[{tname}]")
        norm_rows.append({"transform": tname, "similarity": metric, **a, "mean_AUC_hard": np.mean([a[s] for s in HARD_SETS if s in a]) if a else np.nan})
norm_tbl = pd.DataFrame(norm_rows).sort_values("mean_AUC_hard", ascending=False)
display(norm_tbl.round(4))  # noqa: F821
BEST_NORM = norm_tbl.iloc[0][["transform", "similarity"]].to_dict() if norm_tbl.mean_AUC_hard.notna().any() else {"transform": "sqrt_max_norm", "similarity": "cos"}
report("§22 normalisation", {"best": BEST_NORM, "best mean AUC (hard sets)": float(norm_tbl.mean_AUC_hard.max()),
                              "spread of mean AUC across transforms": float(norm_tbl.mean_AUC_hard.max() - norm_tbl.mean_AUC_hard.min())},
       f"Default spectral preprocessing = '{BEST_NORM['transform']}' with {BEST_NORM['similarity']} similarity, chosen on "
       "same-vs-different separation, not visual appeal.")

## 23. Simple spectral similarity representations

**A. Binned spectra** at several widths up to `MAX_FRAGMENT_MZ` (fraction of peaks beyond reported in the toolkit cell). **B. Peak-aware** greedy cosine and modified cosine at ±`peak_match_tol_da`, on a bounded pair subsample; cross-checked against matchms if installed.

In [ ]:
bt = BEST_NORM["transform"]
bin_rows = []
for w in CONFIG["bin_widths"]:
    Xw = binned_matrix(PEAKS_TRAIN_CLEAN, w, transform=bt, norm="l2")
    PAIRS[f"bin{w}"] = pair_cosine(Xw, PAIRS.i.values, PAIRS.j.values)
    a = auc_by_set(PAIRS, f"bin{w}")
    bin_rows.append({"method": f"binned {w} Da", "n_bins": Xw.shape[1], "density": Xw.nnz / max(1, Xw.shape[0] * Xw.shape[1]), **a})
_sub = PAIRS.groupby("set", group_keys=False).apply(lambda g: g.sample(min(len(g), CONFIG["n_pairs_peakaware"]), random_state=SEED))
PAIRS["peak_cos"] = np.nan
PAIRS["mod_cos"] = np.nan
pm_ = S.precursor_mz.to_numpy()
f_ = INTENSITY_TRANSFORMS[bt]
t0 = time.time()
for k, (i, j) in zip(_sub.index, _sub[["i", "j"]].to_numpy()):
    (m1, i1), (m2, i2) = PEAKS_TRAIN_CLEAN[i], PEAKS_TRAIN_CLEAN[j]
    if len(m1) and len(m2):
        PAIRS.at[k, "peak_cos"] = cosine_greedy(m1, f_(i1), m2, f_(i2))
        PAIRS.at[k, "mod_cos"] = cosine_greedy(m1, f_(i1), m2, f_(i2), precursor_shift=pm_[i] - pm_[j])
    else:
        PAIRS.at[k, "peak_cos"] = PAIRS.at[k, "mod_cos"] = 0.0
print(f"peak-aware similarities: {len(_sub):,} pairs in {time.time()-t0:.1f}s")
for col, name in (("peak_cos", f"greedy cosine ±{CONFIG['peak_match_tol_da']} Da"), ("mod_cos", "modified cosine")):
    bin_rows.append({"method": name, **auc_by_set(PAIRS.dropna(subset=[col]), col)})
sim_tbl = pd.DataFrame(bin_rows)
sim_tbl["mean_AUC_hard"] = sim_tbl[[s for s in HARD_SETS if s in sim_tbl]].mean(axis=1)
display(sim_tbl.round(4))  # noqa: F821

MATCHMS_CHECK = "matchms not installed — skipped"
if OPTIONAL.get("matchms"):
    try:
        from matchms import Spectrum
        from matchms.similarity import CosineGreedy
        cg = CosineGreedy(tolerance=CONFIG["peak_match_tol_da"])
        diffs = []
        for i, j in _sub[["i", "j"]].to_numpy()[:50]:
            (m1, i1), (m2, i2) = PEAKS_TRAIN_CLEAN[i], PEAKS_TRAIN_CLEAN[j]
            if len(m1) < 1 or len(m2) < 1:
                continue
            s1 = Spectrum(mz=m1, intensities=i1 / i1.max(), metadata={"precursor_mz": float(pm_[i])})
            s2 = Spectrum(mz=m2, intensities=i2 / i2.max(), metadata={"precursor_mz": float(pm_[j])})
            diffs.append(abs(float(cg.pair(s1, s2)["score"]) - cosine_greedy(m1, i1, m2, i2)))
        MATCHMS_CHECK = f"max |ours − matchms| over {len(diffs)} pairs = {max(diffs):.2e}"
    except Exception as ex:
        MATCHMS_CHECK = f"matchms check failed: {type(ex).__name__}: {ex}"
best_sim = sim_tbl.sort_values("mean_AUC_hard", ascending=False).iloc[0]
report("§23 similarity representations", {"best method (hard sets)": best_sim.method, "its mean AUC": float(best_sim.mean_AUC_hard),
                                           "MAX_FRAGMENT_MZ": MAX_FRAGMENT_MZ, "peaks beyond MAX_FRAGMENT_MZ": f"{FRAC_PEAKS_BEYOND:.4%}",
                                           "matchms cross-check": MATCHMS_CHECK},
       f"Baseline spectral retrieval representation for later notebooks = {best_sim.method} with '{bt}' intensities.")
BEST_SIM_COL = {"greedy": "peak_cos", "modified": "mod_cos"}.get(best_sim.method.split()[0], f"bin{best_sim.method.split()[1]}" if best_sim.method.startswith("binned") else "cos_prov")

## 24. Same-molecule vs different-molecule similarity

**QUESTION.** How much of spectral similarity reflects *structure* rather than *mass* or *acquisition*? If AUC stays high from N1 to N6, the spectrum carries structural information beyond mass and acquisition; if it collapses at N3, the easy separation was just mass.

In [ ]:
rows = []
Pv = PAIRS.dropna(subset=[BEST_SIM_COL])
pos_s = Pv.loc[Pv.label == 1, BEST_SIM_COL]
for s in NEG_SETS:
    neg = Pv.loc[Pv.set == s, BEST_SIM_COL]
    if len(neg) < 10 or len(pos_s) < 10:
        continue
    sd = np.sqrt((pos_s.var() + neg.var()) / 2)
    rows.append({"negative set": s, "n_neg": len(neg), "mean_pos": pos_s.mean(), "mean_neg": neg.mean(),
                 "median_neg": neg.median(), "AUC": roc_auc_score(np.r_[np.ones(len(pos_s)), np.zeros(len(neg))], np.r_[pos_s, neg]),
                 "cohen_d": (pos_s.mean() - neg.mean()) / sd if sd > 0 else np.nan})
sep_tbl = pd.DataFrame(rows)
display(sep_tbl.round(3))  # noqa: F821
fig, ax = plt.subplots(figsize=(12, 3.6))
bins = np.linspace(0, 1, 41)
ax.hist(pos_s, bins=bins, density=True, histtype="stepfilled", alpha=0.35, color="green", label="POS same connectivity")
for s in NEG_SETS:
    ax.hist(Pv.loc[Pv.set == s, BEST_SIM_COL], bins=bins, density=True, histtype="step", lw=1.3, label=s)
ax.set_xlabel(f"similarity ({BEST_SIM_COL})"); ax.set_ylabel("density"); ax.legend(fontsize=7); ax.set_title("Positive vs progressively harder negatives")
savefig(fig, "24_pair_similarity")
same_add = pos_pairs.get("same_adduct")
adduct_effect = {}
if same_add is not None and pos_pairs["same_adduct"].nunique() == 2:
    pp = PAIRS.loc[PAIRS.label == 1]
    n3 = PAIRS.loc[PAIRS.set == HARD_SETS[0], "cos_prov"]
    for flag, name in ((True, "same adduct"), (False, "different adduct")):
        v_ = pp.loc[pp.same_adduct == flag, "cos_prov"]
        if len(v_) >= 10 and len(n3) >= 10:
            adduct_effect[name] = {"median_cos": float(v_.median()),
                                   "AUC_vs_" + HARD_SETS[0]: float(roc_auc_score(np.r_[np.ones(len(v_)), np.zeros(len(n3))], np.r_[v_, n3]))}
auc = dict(zip(sep_tbl["negative set"], sep_tbl["AUC"]))
report("§24 structure vs acquisition", {"AUC by negative set": {k: round(v, 3) for k, v in auc.items()},
                                        "AUC drop N1 → hardest": float(auc.get(NEG_SETS[0], np.nan) - auc.get(NEG_SETS[-1], np.nan)) if auc else np.nan,
                                        "positives by adduct agreement": adduct_effect},
       "Spectral similarity is evaluated against mass-matched negatives (N3+), because candidates in retrieval are always mass-matched. "
       "N1/N2 AUCs overstate usefulness and must not be used for model selection.")
SEP_TBL, ADDUCT_EFFECT = sep_tbl, adduct_effect

## 25. Molecular descriptor EDA

**QUESTION.** How chemically diverse is the library? One compact quantile table and one Spearman correlation map, rather than a plot per descriptor.

In [ ]:
EDA_DESC = ["exact_mass", "num_heavy_atoms", "num_heteroatoms", "num_rings", "num_aromatic_rings", "tpsa", "logp",
            "h_bond_donors", "h_bond_acceptors", "num_rotatable_bonds", "formal_charge", "fraction_aromatic_atoms"]
desc_q = molecule_df[EDA_DESC].quantile([0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)
display(desc_q)  # noqa: F821
corr = molecule_df[EDA_DESC].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(EDA_DESC))); ax.set_xticklabels(EDA_DESC, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(EDA_DESC))); ax.set_yticklabels(EDA_DESC, fontsize=8)
for (r_, c_), v_ in np.ndenumerate(corr.values):
    ax.text(c_, r_, f"{v_:.1f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax, shrink=0.7); ax.set_title("Spearman correlation of molecule descriptors")
savefig(fig, "25_descriptor_correlation")
_hi = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
report("§25 descriptors", {"molecules": N_MOL, "exact mass p1–p99": f"{desc_q.loc['exact_mass', 0.01]:.0f}–{desc_q.loc['exact_mass', 0.99]:.0f} Da",
                            "strongest descriptor pair |ρ|": f"{_hi.abs().idxmax()} ρ={_hi[_hi.abs().idxmax()]:.2f}",
                            "charged molecules": f"{(molecule_df.formal_charge != 0).mean():.2%}"},
       "Descriptor distributions are the reference for checking whether external candidate libraries (Notebook 02) are chemically comparable to the training library.")

## 26. Murcko scaffold analysis

**QUESTION.** How many structural families exist, and how concentrated are they?

**WHY IT MATTERS.** "Unseen molecule, familiar scaffold" and "unseen molecule, unseen scaffold" are different generalisation regimes; §33 builds a validation split for the second one. Acyclic molecules have an empty scaffold and are reported separately rather than lumped into one giant "family".

In [ ]:
scaf = molecule_df["murcko_scaffold"].fillna("<error>")
acyclic = scaf == ""
fam = scaf[~acyclic].value_counts()
scaffold_df = pd.DataFrame({"murcko_scaffold": fam.index, "n_molecules": fam.values})
scaffold_df["generic_scaffold"] = scaffold_df.murcko_scaffold.map(molecule_df.drop_duplicates("murcko_scaffold").set_index("murcko_scaffold")["scaffold_smiles"])
report("§26 scaffolds", {"unique structures": N_MOL, "acyclic (no scaffold)": f"{acyclic.sum()} ({acyclic.mean():.1%})",
                         "unique Murcko scaffolds": len(fam), "unique generic scaffolds": molecule_df.loc[~acyclic, "scaffold_smiles"].nunique(),
                         "molecules per scaffold p50/p90/max": f"{fam.median():.0f} / {fam.quantile(.9):.0f} / {fam.max()}",
                         "singleton scaffold fraction (of scaffolds)": f"{(fam == 1).mean():.1%}",
                         "molecules in singleton scaffolds": f"{fam[fam == 1].sum() / N_MOL:.1%}",
                         "largest families": fam.head(5).to_dict()},
       "scaffold_metadata.parquet saved (§37); scaffold-grouped folds built in §33.")
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.loglog(np.arange(1, len(fam) + 1), fam.values, ".", ms=3)
ax.set_xlabel("scaffold rank"); ax.set_ylabel("molecules"); ax.set_title("Scaffold family size (rank-frequency)")
savefig(fig, "26_scaffold_rank_frequency")
try:
    from rdkit.Chem import Draw
    top_sc = fam.head(12)
    img = Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in top_sc.index], molsPerRow=6, subImgSize=(220, 170),
                               legends=[f"n={n}" for n in top_sc.values])
    img_path = FIGURE_ROOT / "26_top_scaffolds.png"
    img.save(img_path) if hasattr(img, "save") else open(img_path, "wb").write(img.data)
    FIGURES.append(str(img_path))
    display(img)  # noqa: F821
except Exception as ex:
    print("scaffold drawing skipped:", type(ex).__name__, ex)

## 27. Structural similarity landscape

Morgan fingerprints (radius 2, 2048 bits, RDKit `rdFingerprintGenerator`). For a seeded sample of query molecules, nearest-other-molecule Tanimoto and top-5 mean against the **whole** library (exact, via `BulkTanimotoSimilarity`). We also measure the most similar *mass-isobaric* neighbour (±`pair_mass_tol_ppm`): these are the hard negatives a ranker will face.

In [ ]:
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
t0 = time.time()
FPS = []
for s in molecule_df.representative_smiles:
    m = _mol_from_smiles(s)
    FPS.append(_gen.GetFingerprint(m) if m is not None else DataStructs.ExplicitBitVect(2048))
print(f"fingerprints: {len(FPS):,} in {time.time()-t0:.1f}s")
fp_bits = np.zeros((len(FPS), 2048), dtype=np.uint8)
for i, f in enumerate(FPS):
    DataStructs.ConvertToNumpyArray(f, fp_bits[i])
FP_PACKED = np.packbits(fp_bits, axis=1)
del fp_bits

rng = np.random.default_rng(SEED)
q_idx = np.sort(rng.choice(N_MOL, size=min(N_MOL, CONFIG["nn_query_sample"]), replace=False))
masses = molecule_df.exact_mass.to_numpy()
m_order = np.argsort(masses)
m_sorted = masses[m_order]
nn1, top5, iso_max, iso_n = [], [], [], []
t0 = time.time()
for q in q_idx:
    sims = np.asarray(DataStructs.BulkTanimotoSimilarity(FPS[q], FPS))
    sims[q] = -1
    part = np.partition(sims, -5)[-5:] if len(sims) > 5 else sims
    nn1.append(sims.max()); top5.append(part.mean())
    lo_, hi_ = np.searchsorted(m_sorted, [masses[q] * (1 - CONFIG["pair_mass_tol_ppm"] * 1e-6), masses[q] * (1 + CONFIG["pair_mass_tol_ppm"] * 1e-6)])
    iso = m_order[lo_:hi_]
    iso = iso[iso != q]
    iso_n.append(len(iso)); iso_max.append(sims[iso].max() if len(iso) else np.nan)
print(f"NN search: {len(q_idx):,} queries × {N_MOL:,} in {time.time()-t0:.1f}s")
nn_df = pd.DataFrame({"connectivity_key": molecule_df.connectivity_key.values[q_idx], "nn1_tanimoto": nn1, "top5_mean_tanimoto": top5,
                      "n_isobaric": iso_n, "max_tanimoto_isobaric": iso_max})
cats = pd.cut(nn_df.nn1_tanimoto, [-1, 0.3, 0.6, 0.9, 0.999, 1.01], labels=["isolated (<0.3)", "moderate (0.3–0.6)", "crowded (0.6–0.9)", "near-duplicate (0.9–<1)", "identical bits (1.0)"])
fig, ax = plt.subplots(figsize=(10, 3.3))
ax.hist(nn_df.nn1_tanimoto, bins=50, alpha=0.7, label="nearest other molecule")
ax.hist(nn_df.max_tanimoto_isobaric.dropna(), bins=50, alpha=0.6, label=f"nearest isobaric (±{CONFIG['pair_mass_tol_ppm']:g} ppm)")
ax.set_xlabel("Tanimoto (Morgan r=2, 2048)"); ax.legend(); ax.set_title("Structural neighbourhoods")
savefig(fig, "27_nn_tanimoto")
report("§27 similarity landscape", {"NN Tanimoto p10/p50/p90": " / ".join(f"{v:.2f}" for v in nn_df.nn1_tanimoto.quantile([.1, .5, .9])),
                                    "categories": cats.value_counts(normalize=True).round(3).to_dict(),
                                    "queries with ≥1 isobaric molecule": f"{(nn_df.n_isobaric > 0).mean():.1%}",
                                    "nearest isobaric Tanimoto p50": float(nn_df.max_tanimoto_isobaric.median())},
       "Fingerprints saved (packed bits). Isobaric neighbours with high Tanimoto define the hard-negative pool for ranking models.")

## 28. Formula ambiguity

**QUESTION.** If the molecular formula were known perfectly, how many structures would remain? This is the ceiling of formula-based filtering *within this library*.

In [ ]:
fstat = molecule_df.groupby("molecular_formula").agg(n_structures=("connectivity_key", "size"), n_scaffolds=("murcko_scaffold", "nunique"))
per_mol_formula = molecule_df.molecular_formula.map(fstat.n_structures)
report("§28 formula ambiguity", {"unique formulas": len(fstat),
                                 "structures per formula (per formula) p50/p90/p95/max": " / ".join(f"{v:.0f}" for v in fstat.n_structures.quantile([.5, .9, .95, 1])),
                                 "structures sharing MY formula (per molecule) p50/p90/p95/max": " / ".join(f"{v:.0f}" for v in per_mol_formula.quantile([.5, .9, .95, 1])),
                                 "molecules with a unique formula": f"{(per_mol_formula == 1).mean():.1%}",
                                 "scaffolds per formula p90": float(fstat.n_scaffolds.quantile(.9))},
       "Formula alone does not identify structure for molecules in crowded formulas; spectra must discriminate among isomers.")
display(fstat.sort_values("n_structures", ascending=False).head(10))  # noqa: F821

## 29. Mass ambiguity / candidate-space analysis (mandatory before candidate generation)

**QUESTION.** How much does accurate neutral mass shrink the search space? For every molecule: number of *other* connectivities within ±1, 2, 5, 10, 20 ppm of its exact mass (vectorised `searchsorted` on sorted masses — exact counts, no sampling).

**Caveat that shapes Notebook 02.** These counts are *within the training library*. Under connectivity-grouped CV (and on test) the true structure is by construction absent from the training library, so the real candidate space must come from an external structure database; the counts here are a lower bound on crowding.

In [ ]:
amb = molecule_df[["connectivity_key", "exact_mass", "molecular_formula", "num_heavy_atoms"]].copy()
for t in CONFIG["ambiguity_tols_ppm"]:
    lo_ = np.searchsorted(m_sorted, masses * (1 - t * 1e-6), "left")
    hi_ = np.searchsorted(m_sorted, masses * (1 + t * 1e-6), "right")
    amb[f"n_within_{t}ppm"] = (hi_ - lo_ - 1).astype(np.int32)
amb["n_same_formula"] = (per_mol_formula - 1).astype(np.int32).values
amb_q = pd.DataFrame([{"tolerance": f"±{t} ppm", **{k: amb[f"n_within_{t}ppm"].quantile(q) for k, q in
                       (("median", .5), ("p75", .75), ("p90", .9), ("p95", .95), ("p99", .99), ("max", 1))},
                       "zero-ambiguity %": 100 * (amb[f"n_within_{t}ppm"] == 0).mean()} for t in CONFIG["ambiguity_tols_ppm"]])
display(amb_q.round(1))  # noqa: F821
amb["mass_range"] = pd.cut(amb.exact_mass, [0, 200, 300, 400, 500, 700, 1000, np.inf])
amb["complexity"] = pd.cut(amb.num_heavy_atoms, [0, 15, 25, 35, 50, 1e9], labels=["≤15", "16–25", "26–35", "36–50", ">50"])
rt = f"n_within_{int(RECOMMENDED_PPM)}ppm" if f"n_within_{int(RECOMMENDED_PPM)}ppm" in amb else f"n_within_{CONFIG['ambiguity_tols_ppm'][-2]}ppm"
by_mass = amb.groupby("mass_range", observed=True)[rt].describe(percentiles=[.5, .9])[["count", "50%", "90%", "max"]]
by_cplx = amb.groupby("complexity", observed=True)[rt].describe(percentiles=[.5, .9])[["count", "50%", "90%", "max"]]
print(f"Candidates within recommended tolerance ({rt}) by mass range:"); display(by_mass)  # noqa: F821
print("…by structural complexity (heavy atoms):"); display(by_cplx)  # noqa: F821
iso_share = (amb.n_same_formula / amb[rt].replace(0, np.nan)).median()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
axes[0].plot(CONFIG["ambiguity_tols_ppm"], amb_q["median"], "o-", label="median"); axes[0].plot(CONFIG["ambiguity_tols_ppm"], amb_q["p90"], "s--", label="p90")
axes[0].set_xscale("log"); axes[0].set_xlabel("tolerance (ppm)"); axes[0].set_ylabel("other connectivities in window"); axes[0].legend(); axes[0].set_title("Mass-window crowding (training library)")
axes[1].scatter(amb.exact_mass, amb[rt] + 1, s=2, alpha=0.3); axes[1].set_yscale("log"); axes[1].set_xlabel("exact mass"); axes[1].set_ylabel(f"{rt} + 1")
axes[1].set_title("Crowding vs mass")
savefig(fig, "29_mass_ambiguity")

# test coverage by the training library (lower bound on what a train-only library could offer)
tu = test_meta.dropna(subset=["neutral_mass_obs"]).groupby("unit_id", observed=True)["neutral_mass_obs"].median()
if len(tu):
    lo_ = np.searchsorted(m_sorted, tu.values * (1 - RECOMMENDED_PPM * 1e-6), "left")
    hi_ = np.searchsorted(m_sorted, tu.values * (1 + RECOMMENDED_PPM * 1e-6), "right")
    test_cov = pd.Series(hi_ - lo_, index=tu.index)
else:
    test_cov = pd.Series(dtype=int)
report("§29 mass ambiguity", {"window used for summary": rt,
                              "median / p90 candidates at recommended tol (train library)": f"{amb[rt].median():.0f} / {amb[rt].quantile(.9):.0f}",
                              "median share of window explained by exact isomers": float(iso_share),
                              "test units with ≥1 train-library molecule in window": f"{(test_cov > 0).mean():.1%}" if len(test_cov) else "n/a",
                              "test units: median train-library candidates": float(test_cov.median()) if len(test_cov) else np.nan,
                              "test units without reconstructable neutral mass": int(test_meta.unit_id.nunique() - len(tu))},
       "mass_ambiguity.parquet saved. Candidate generation starts from neutral mass, but the training library cannot be the candidate "
       "source for unseen structures; Notebook 02 must attach an external library and measure recall under grouped CV.")
AMB, AMB_Q, TEST_COV = amb, amb_q, test_cov
CHECKS["Candidate ambiguity measured"] = True

## 30. Train/test distribution shift

**QUESTION.** Which observable variables differ between train and test, and by how much?

Numeric: KS statistic and Wasserstein distance (divided by the train IQR so that variables are comparable). Categorical: total variation distance and Jensen–Shannon distance (base 2). Spectral features use the §18 samples. Severity thresholds (KS / TVD / JSD): < 0.05 negligible, < 0.1 small, < 0.2 moderate, ≥ 0.2 large.

In [ ]:
def severity(v):
    return "n/a" if not np.isfinite(v) else "negligible" if v < 0.05 else "small" if v < 0.1 else "moderate" if v < 0.2 else "large"


def numeric_shift(a, b):
    a, b = pd.Series(a).dropna().to_numpy(float), pd.Series(b).dropna().to_numpy(float)
    if len(a) < 20 or len(b) < 20:
        return np.nan, np.nan
    iqr = np.subtract(*np.quantile(a, [.75, .25])) or 1.0
    return stats.ks_2samp(a, b).statistic, stats.wasserstein_distance(a, b) / iqr


def categorical_shift(a, b):
    pa_, pb_ = pd.Series(a).astype(object).fillna("<na>").value_counts(normalize=True), pd.Series(b).astype(object).fillna("<na>").value_counts(normalize=True)
    idx = pa_.index.union(pb_.index)
    pa_, pb_ = pa_.reindex(idx, fill_value=0).to_numpy(), pb_.reindex(idx, fill_value=0).to_numpy()
    return 0.5 * np.abs(pa_ - pb_).sum(), float(jensenshannon(pa_, pb_, base=2))


shift_rows = []
num_vars = [("precursor m/z", train_meta.precursor_mz, test_meta.precursor_mz),
            ("neutral mass (reconstructed)", train_meta.neutral_mass_obs, test_meta.neutral_mass_obs),
            ("n_peaks", train_s.n_peaks, test_s.n_peaks), ("spectral entropy", train_s.spectral_entropy, test_s.spectral_entropy),
            ("fragment mass range", train_s.fragment_mass_range, test_s.fragment_mass_range),
            ("fragment coverage", train_s.fragment_coverage, test_s.fragment_coverage)]
if "ce_unit" in train_meta and "ce_unit" in test_meta:
    for u in train_meta.ce_unit.value_counts().index[:3]:
        if not str(u).startswith(("missing", "unparsed")):
            num_vars.append((f"collision energy [{u}]", train_meta.loc[train_meta.ce_unit == u, "ce_value"], test_meta.loc[test_meta.ce_unit == u, "ce_value"]))
if "n_peaks_all" in train_meta and train_meta.n_peaks_all.notna().any():
    num_vars.append(("n_peaks (population)", train_meta.n_peaks_all, test_meta.n_peaks_all))
for name, a, b in num_vars:
    ks, wd = numeric_shift(a, b)
    shift_rows.append({"variable": name, "type": "numeric", "metric": "KS", "value": ks, "secondary (W/IQR)": wd, "severity": severity(ks),
                       "train median": np.nanmedian(a) if len(a) else np.nan, "test median": np.nanmedian(b) if len(b) else np.nan})
for c in ("adduct", "ion_mode", "instrument", "source", "ce_unit"):
    if c in train_meta and c in test_meta:
        tvd, jsd = categorical_shift(train_meta[c], test_meta[c])
        top_tr = train_meta[c].astype(object).value_counts(normalize=True).head(1)
        top_te = test_meta[c].astype(object).value_counts(normalize=True).head(1)
        shift_rows.append({"variable": c, "type": "categorical", "metric": "TVD", "value": tvd, "secondary (JSD)": jsd, "severity": severity(tvd),
                           "train median": f"{top_tr.index[0]} {top_tr.iloc[0]:.0%}", "test median": f"{top_te.index[0]} {top_te.iloc[0]:.0%}"})
    elif c in train_meta:
        shift_rows.append({"variable": c, "type": "categorical", "metric": "—", "value": np.nan, "severity": "not observable in test"})
shift_tbl = pd.DataFrame(shift_rows)


def _interp(r):
    if r["severity"] in ("negligible", "small"):
        return "comparable"
    if r["severity"] == "not observable in test":
        return "cannot be used as a test-time input"
    return f"{r['severity']} shift: train {r.get('train median')} vs test {r.get('test median')}"


shift_tbl["interpretation"] = shift_tbl.apply(_interp, axis=1)
display(shift_tbl.round(4))  # noqa: F821
large = shift_tbl[shift_tbl.severity.isin(["moderate", "large"])].variable.tolist()
report("§30 train/test shift", {"variables with moderate/large shift": large or "none",
                                "not observable in test": shift_tbl.loc[shift_tbl.severity == "not observable in test", "variable"].tolist()},
       "Shifted variables must be explicit model inputs or be reweighted; validation should include a test-like diagnostic (§31/§34) when a source matches test better.")
CHECKS["Train/test shift quantified"] = len(shift_tbl) > 0

## 31. Which training source looks most like test?

Every source is compared with test on the same variables; a composite is the mean of the per-variable distances (KS for numeric, TVD for categorical). A source is called **most test-like** only if its composite is at least 20 % lower than the runner-up — otherwise the table says there is no clear winner.

In [ ]:
TESTLIKE_SOURCE = None
if "source" in train_meta and train_meta.source.nunique() > 1:
    src_rows = []
    for s in train_meta.source.value_counts().index:
        trm, trs = train_meta[train_meta.source == s], train_s[train_s.source == s]
        if len(trm) < 50:
            continue
        row = {"source": s, "n_spectra": len(trm)}
        row["precursor KS"] = numeric_shift(trm.precursor_mz, test_meta.precursor_mz)[0]
        row["n_peaks KS"] = numeric_shift(trs.n_peaks, test_s.n_peaks)[0]
        row["entropy KS"] = numeric_shift(trs.spectral_entropy, test_s.spectral_entropy)[0]
        for c in ("adduct", "ion_mode", "instrument", "ce_unit"):
            if c in trm and c in test_meta:
                row[f"{c} TVD"] = categorical_shift(trm[c], test_meta[c])[0]
        src_rows.append(row)
    src_tbl = pd.DataFrame(src_rows).set_index("source")
    dist_cols = [c for c in src_tbl.columns if c != "n_spectra"]
    src_tbl["composite"] = src_tbl[dist_cols].mean(axis=1)
    src_tbl = src_tbl.sort_values("composite")
    display(src_tbl.round(3))  # noqa: F821
    if len(src_tbl) >= 2 and src_tbl.composite.iloc[0] <= 0.8 * src_tbl.composite.iloc[1]:
        TESTLIKE_SOURCE = src_tbl.index[0]
    report("§31 source vs test", {"ranking (composite distance)": src_tbl.composite.round(3).to_dict(), "most test-like": TESTLIKE_SOURCE or "no clear winner (<20% margin)"},
           "A test-like holdout (§34) is built only if a clear winner exists.")
else:
    src_tbl = pd.DataFrame()
    report("§31 source vs test", "No multi-source information in training data; comparison not applicable.")

## 32. Leakage-safe primary cross-validation (permanent)

**QUESTION.** How do we split so that no validation structure is ever seen in training?

**ANALYSIS.** A custom balanced grouped split at `connectivity_key` level. Groups are stratified by neutral-mass decile × dominant source (legitimate, label-free statistics of the group) and, within each stratum, assigned largest-first to the fold with the fewest spectra (ties broken by a seeded shuffle). This balances molecules, spectra, mass and source simultaneously. Spectra with invalid structures receive `fold = -1` (reported, not dropped).

In [ ]:
def build_grouped_folds(groups: pd.DataFrame, key_col, weight_col, strata_col=None, n_splits=5, seed=SEED):
    g = groups[[key_col, weight_col] + ([strata_col] if strata_col else [])].copy()
    g["_r"] = np.random.default_rng(seed).random(len(g))
    g["_s"] = g[strata_col].astype(str) if strata_col else "all"
    g = g.sort_values(["_s", weight_col, "_r"], ascending=[True, False, True])
    glob = np.zeros(n_splits)
    fold = np.empty(len(g), dtype=np.int8)
    w = g[weight_col].to_numpy(float)
    pos = 0
    for _, n in g.groupby("_s", sort=True).size().items():
        loc = np.zeros(n_splits)
        for k in range(pos, pos + n):
            f = int(np.lexsort((glob, loc))[0])
            fold[k] = f
            loc[f] += w[k]
            glob[f] += w[k]
        pos += n
    return pd.Series(fold, index=g[key_col].to_numpy(), name="fold")


mol_groups = per_mol.reset_index()[["connectivity_key", "n_spectra"]].merge(molecule_df[["connectivity_key", "exact_mass"]], on="connectivity_key")
mol_groups["mass_decile"] = pd.qcut(mol_groups.exact_mass, 10, labels=False, duplicates="drop")
if "source" in train_meta:
    dom = train_meta.dropna(subset=["connectivity_key"]).groupby("connectivity_key", observed=True)["source"].agg(lambda s: s.value_counts().index[0])
    top_sources = set(train_meta.source.value_counts().head(5).index)
    mol_groups["dom_source"] = mol_groups.connectivity_key.map(dom).map(lambda s: s if s in top_sources else "other")
    mol_groups["stratum"] = mol_groups.mass_decile.astype(str) + "|" + mol_groups.dom_source.astype(str)
else:
    mol_groups["stratum"] = mol_groups.mass_decile.astype(str)
fold_of = build_grouped_folds(mol_groups, "connectivity_key", "n_spectra", "stratum", CONFIG["n_folds"], SEED)
connectivity_folds = pd.DataFrame({"connectivity_key": fold_of.index, "fold": fold_of.values}).merge(mol_groups[["connectivity_key", "n_spectra"]], on="connectivity_key")
train_meta["fold"] = train_meta.connectivity_key.astype(object).map(fold_of).fillna(-1).astype(np.int8)

# ---- leakage assertions
overlaps = {}
for f in range(CONFIG["n_folds"]):
    tr = set(train_meta.loc[(train_meta.fold != f) & (train_meta.fold >= 0), "connectivity_key"].astype(object))
    va = set(train_meta.loc[train_meta.fold == f, "connectivity_key"].astype(object))
    overlaps[f] = len(tr & va)
print("train ∩ validation connectivity per fold:", overlaps)
assert all(v == 0 for v in overlaps.values()), "LEAKAGE: connectivity shared between train and validation"
assert connectivity_folds.connectivity_key.is_unique, "a connectivity was assigned to more than one fold"
assert (train_meta.dropna(subset=["connectivity_key"]).groupby("connectivity_key", observed=True).fold.nunique() == 1).all()
check("Connectivity CV built", len(connectivity_folds) == N_MOL and set(connectivity_folds.fold) == set(range(CONFIG["n_folds"])))
check("CV leakage = 0", all(v == 0 for v in overlaps.values()), f"overlaps {overlaps}")


def fold_balance(meta, fold_col):
    rows = []
    for f, d in meta[meta[fold_col] >= 0].groupby(fold_col):
        r = {"fold": f, "molecules": d.connectivity_key.nunique(), "spectra": len(d), "precursor median": d.precursor_mz.median()}
        for c in ("adduct", "source"):
            if c in meta:
                r[f"{c} TVD vs all"] = categorical_shift(d[c], meta.loc[meta[fold_col] >= 0, c])[0]
        rows.append(r)
    return pd.DataFrame(rows).set_index("fold")


bal = fold_balance(train_meta, "fold")
display(bal.round(4))  # noqa: F821
report("§32 primary CV", {"folds": CONFIG["n_folds"], "spectra with fold=-1 (invalid label)": int((train_meta.fold < 0).sum()),
                          "molecules per fold (min–max)": f"{bal.molecules.min()}–{bal.molecules.max()}",
                          "spectra per fold (min–max)": f"{bal.spectra.min()}–{bal.spectra.max()}",
                          "max adduct TVD vs all": float(bal.filter(like="adduct").max().max()) if bal.filter(like="adduct").size else np.nan},
       "connectivity_folds.parquet is the PERMANENT primary split. Every later notebook must load it; no notebook may re-split.")

## 33. Secondary hard validation — unseen scaffold family

Same builder, grouped by Murcko scaffold (acyclic molecules form their own singleton groups, keyed by connectivity). This answers a different question — generalisation to unseen structural families — and **does not replace** the primary CV. Difficulty is measured directly: the Tanimoto of validation molecules to their nearest training molecule under each scheme.

In [ ]:
mg = mol_groups.merge(molecule_df[["connectivity_key", "murcko_scaffold"]], on="connectivity_key")
mg["scaffold_group"] = np.where(mg.murcko_scaffold.fillna("") == "", "ACYCLIC::" + mg.connectivity_key, mg.murcko_scaffold)
sg = mg.groupby("scaffold_group").agg(n_spectra=("n_spectra", "sum"), mass=("exact_mass", "median")).reset_index()
sg["mass_decile"] = pd.qcut(sg.mass, 10, labels=False, duplicates="drop")
sfold = build_grouped_folds(sg, "scaffold_group", "n_spectra", "mass_decile", CONFIG["n_folds"], SEED + 7)
mg["scaffold_fold"] = mg.scaffold_group.map(sfold).astype(np.int8)
train_meta["scaffold_fold"] = train_meta.connectivity_key.astype(object).map(mg.set_index("connectivity_key").scaffold_fold).fillna(-1).astype(np.int8)
sc_overlap = {f: len(set(mg.loc[mg.scaffold_fold != f, "scaffold_group"]) & set(mg.loc[mg.scaffold_fold == f, "scaffold_group"])) for f in range(CONFIG["n_folds"])}
assert all(v == 0 for v in sc_overlap.values())

key_to_row = pd.Series(np.arange(N_MOL), index=molecule_df.connectivity_key)
def nn_to_train(fold_series, f, n=500):
    va = key_to_row[fold_series[fold_series == f].index].to_numpy()
    tr = key_to_row[fold_series[fold_series != f].index].to_numpy()
    va = np.random.default_rng(SEED).choice(va, size=min(n, len(va)), replace=False)
    tr_fps = [FPS[i] for i in tr]
    return np.array([max(DataStructs.BulkTanimotoSimilarity(FPS[i], tr_fps)) for i in va]) if len(tr_fps) else np.array([np.nan])


nn_primary = nn_to_train(connectivity_folds.set_index("connectivity_key").fold, 0)
nn_scaffold = nn_to_train(mg.set_index("connectivity_key").scaffold_fold, 0)
scaffold_folds = mg[["connectivity_key", "scaffold_group", "scaffold_fold"]]
report("§33 scaffold CV", {"scaffold groups": len(sg), "scaffold overlap per fold": sc_overlap,
                           "val→nearest-train Tanimoto median: primary CV": float(np.median(nn_primary)),
                           "val→nearest-train Tanimoto median: scaffold CV": float(np.median(nn_scaffold))},
       "scaffold_folds saved; used as a secondary 'unseen family' diagnostic, never for model selection alone.")

## 34. Optional test-like validation

Built only if §31 found a clearly most test-like source. Validation = that source's spectra; training = everything else **minus any connectivity that appears in validation** (so it remains leakage-free).

In [ ]:
if TESTLIKE_SOURCE is not None:
    va_keys = set(train_meta.loc[train_meta.source == TESTLIKE_SOURCE, "connectivity_key"].dropna().astype(object))
    testlike = pd.DataFrame({"spectrum_id": train_meta.spectrum_id,
                             "role": np.where(train_meta.source == TESTLIKE_SOURCE, "validation",
                                              np.where(train_meta.connectivity_key.astype(object).isin(va_keys), "excluded_shared_structure", "train"))})
    assert not (set(train_meta.loc[testlike.role.values == "train", "connectivity_key"].astype(object)) & va_keys)
    save_artifact(testlike, "testlike_split", extra={"source": str(TESTLIKE_SOURCE)})
    report("§34 test-like split", testlike.role.value_counts().to_dict(), f"Secondary diagnostic only: hold out source '{TESTLIKE_SOURCE}'.")
else:
    report("§34 test-like split", "Not built: no training source is quantitatively the most test-like (§31).")

## 35. Official project ranking metrics

`evaluate_ranking()` compares **connectivity keys**, deduplicates each candidate list by connectivity (first occurrence wins) *before* truncating to 25, and always divides by the total number of queries: a query whose true structure is absent contributes RR = 0 and Hit@k = 0. `candidate_recall` is measured on the full (untruncated) candidate list, so it isolates candidate generation from ranking.

In [ ]:
def _dedup(keys):
    seen, out = set(), []
    for k in keys:
        if k is None or k in seen:
            continue
        seen.add(k)
        out.append(k)
    return out


def evaluate_ranking(true_keys, ranked_keys, k=25, ks=(1, 5, 10, 25)):
    """true_keys: list[str|None]; ranked_keys: list[list[str]] (full candidate lists, best first)."""
    assert len(true_keys) == len(ranked_keys), "one candidate list per query is required"
    n = len(true_keys)
    rr, hits, rec = np.zeros(n), {kk: np.zeros(n) for kk in ks}, np.zeros(n)
    for q, (t, cands) in enumerate(zip(true_keys, ranked_keys)):
        full = _dedup(cands or [])
        if t is not None and t in full:
            rec[q] = 1
            r = full.index(t) + 1
            if r <= k:
                rr[q] = 1 / r
            for kk in ks:
                hits[kk][q] = float(r <= kk)
    out = {f"MRR@{k}": rr.mean() if n else np.nan, **{f"Hit@{kk}": h.mean() if n else np.nan for kk, h in hits.items()},
           "candidate_recall": rec.mean() if n else np.nan, "n_queries": n}
    return out


_key_cache = {}
def smiles_to_key(s):
    if s not in _key_cache:
        _key_cache[s] = competition_connectivity_key(s)
    return _key_cache[s]


def evaluate_smiles_ranking(true_smiles, ranked_smiles, k=25):
    return evaluate_ranking([smiles_to_key(s) for s in true_smiles], [[smiles_to_key(c) for c in (cs or [])] for cs in ranked_smiles], k)


# deterministic tests
filler = [f"K{i:03d}" for i in range(40)]
T = ["A", "B", "C", "D", "E"]
R = [["A"] + filler[:24],                  # rank 1
     [filler[0], "B"] + filler[1:23],      # rank 2
     filler[:24] + ["C"],                  # rank 25
     filler[:25],                          # absent
     filler[:25] + ["E"]]                  # rank 26: outside top-25 but in candidate set
m = evaluate_ranking(T, R)
exp_mrr = (1 + 1 / 2 + 1 / 25 + 0 + 0) / 5
METRIC_TESTS = {
    "MRR@25 exact": abs(m["MRR@25"] - exp_mrr) < 1e-12,
    "Hit@1 = 1/5": abs(m["Hit@1"] - 0.2) < 1e-12,
    "Hit@5 = 2/5": abs(m["Hit@5"] - 0.4) < 1e-12,
    "Hit@10 = 2/5": abs(m["Hit@10"] - 0.4) < 1e-12,
    "Hit@25 = 3/5": abs(m["Hit@25"] - 0.6) < 1e-12,
    "candidate_recall = 4/5": abs(m["candidate_recall"] - 0.8) < 1e-12,
    "denominator includes missing & empty": evaluate_ranking(["A", "B"], [["A"], []])["MRR@25"] == 0.5,
    "dedup by connectivity before ranking": evaluate_ranking(["Y"], [["X", "X", "Y"]])["MRR@25"] == 0.5,
    "SMILES-level: stereo duplicate collapses": abs(evaluate_smiles_ranking(["CCO"], [["C[C@H](N)C(=O)O", "C[C@@H](N)C(=O)O", "OCC"]])["MRR@25"] - 0.5) < 1e-12,
    "unknown true key scores 0": evaluate_ranking([None], [["A"]])["MRR@25"] == 0.0,
}
for k_, v_ in METRIC_TESTS.items():
    print(f"  [{'PASS' if v_ else 'FAIL'}] {k_}")
print("  metrics:", {k_: round(v_, 4) for k_, v_ in m.items()})
check("Ranking metric tests pass", all(METRIC_TESTS.values()))

## 36. Submission validation utility

The format is inferred from `sample_submission` (long format, wide rank columns, or one delimited column), then enforced. Unit tests feed deliberately broken submissions and require each to fail.

In [ ]:
def infer_submission_format(sub):
    id_col = SUB_ID_COL
    other = [c for c in sub.columns if c != id_col]
    if not sub[id_col].is_unique:
        rank_col = next((c for c in other if "rank" in c.lower()), None)
        smi_col = next((c for c in other if c != rank_col), None)
        return {"kind": "long", "id_col": id_col, "rank_col": rank_col, "smiles_col": smi_col}
    # rank/score/confidence columns are bookkeeping, never predictions
    preds = [c for c in other if not re.search(r"rank|score|prob|conf|weight", c, re.I)] or other
    if len(preds) > 1:
        return {"kind": "wide", "id_col": id_col, "pred_cols": preds}
    col = preds[0]
    vals = sub[col].dropna().astype(str)
    delim = " "
    for d in ("|", ";", ",", "\t", " "):
        if vals.str.contains(re.escape(d)).any():
            delim = d
            break
    return {"kind": "delimited", "id_col": id_col, "pred_col": col, "delimiter": delim}


SUB_FORMAT = infer_submission_format(sample_sub)
print("inferred submission format:", SUB_FORMAT)


def submission_to_lists(df, fmt=None):
    fmt = fmt or SUB_FORMAT
    if fmt["kind"] == "long":
        d = df.sort_values([fmt["id_col"], fmt["rank_col"]]) if fmt["rank_col"] else df
        return d.groupby(fmt["id_col"], sort=False)[fmt["smiles_col"]].apply(lambda s: [x for x in s if isinstance(x, str) and x.strip()]).to_dict()
    if fmt["kind"] == "wide":
        return {r[fmt["id_col"]]: [x for x in (r[c] for c in fmt["pred_cols"]) if isinstance(x, str) and x.strip()] for _, r in df.iterrows()}
    return {r[fmt["id_col"]]: ([t for t in str(r[fmt["pred_col"]]).split(fmt["delimiter"]) if t.strip()] if isinstance(r[fmt["pred_col"]], str) else [])
            for _, r in df.iterrows()}


def validate_submission(df, check_smiles=True, fmt=None, expected_ids=None, max_k=None):
    fmt, max_k = fmt or SUB_FORMAT, max_k or CONFIG["max_candidates"]
    expected_ids = set(map(str, expected_ids if expected_ids is not None else sample_sub[SUB_ID_COL]))
    errors, warns_ = [], []
    need = {fmt["id_col"]} | ({fmt["pred_col"]} if fmt["kind"] == "delimited" else set(fmt.get("pred_cols", [])) if fmt["kind"] == "wide" else {fmt["smiles_col"]})
    if set(df.columns) != set(sample_sub.columns):
        errors.append(f"columns {list(df.columns)} ≠ expected {list(sample_sub.columns)}")
        if not need <= set(df.columns):
            return {"valid": False, "errors": errors, "warnings": warns_}
    if fmt["kind"] != "long" and df[fmt["id_col"]].duplicated().any():
        errors.append(f"{int(df[fmt['id_col']].duplicated().sum())} duplicate ids")
    ids = set(df[fmt["id_col"]].astype(str))
    if ids != expected_ids:
        errors.append(f"id mismatch: missing {len(expected_ids - ids)}, unexpected {len(ids - expected_ids)}")
    lists = submission_to_lists(df, fmt)
    empty = [k for k, v in lists.items() if not v]
    if empty:
        errors.append(f"{len(empty)} ids with null/empty prediction")
    too_long = [k for k, v in lists.items() if len(v) > max_k]
    if too_long:
        errors.append(f"{len(too_long)} ids with > {max_k} candidates")
    if check_smiles:
        n_bad, n_dup = 0, 0
        for v in lists.values():
            keys = [smiles_to_key(s) for s in v]
            n_bad += sum(k is None for k in keys)
            kk = [k for k in keys if k is not None]
            n_dup += len(kk) - len(set(kk))
        if n_bad:
            errors.append(f"{n_bad} invalid SMILES")
        if n_dup:
            warns_.append(f"{n_dup} connectivity duplicates (wasted ranks)")
    return {"valid": not errors, "errors": errors, "warnings": warns_}


def _with_aux_cols(df):
    """Carry non-prediction sample columns (e.g. a constant rank/score column) so the column set matches."""
    for c in sample_sub.columns:
        if c not in df:
            df[c] = sample_sub[c].iloc[0] if len(sample_sub) else None
    return df[list(sample_sub.columns)]


def _make_sub(ids, lists):
    if SUB_FORMAT["kind"] == "delimited":
        return _with_aux_cols(pd.DataFrame({SUB_ID_COL: ids, SUB_FORMAT["pred_col"]: [SUB_FORMAT["delimiter"].join(l) if l is not None else None for l in lists]}))
    if SUB_FORMAT["kind"] == "wide":
        return _with_aux_cols(pd.DataFrame([{SUB_ID_COL: i, **{c: (l[k] if l and k < len(l) else None) for k, c in enumerate(SUB_FORMAT["pred_cols"])}} for i, l in zip(ids, lists)]))
    rows = [{SUB_ID_COL: i, SUB_FORMAT["rank_col"]: r + 1, SUB_FORMAT["smiles_col"]: s} for i, l in zip(ids, lists) for r, s in enumerate(l or [None])]
    return pd.DataFrame(rows)[list(sample_sub.columns)]


_ids = sample_sub[SUB_ID_COL].drop_duplicates().tolist()
_good = ["CCO", "c1ccccc1O", "CC(=O)O"]
sub_tests = {
    "valid synthetic submission passes": validate_submission(_make_sub(_ids, [_good] * len(_ids)))["valid"],
    "duplicate id fails": SUB_FORMAT["kind"] == "long" or not validate_submission(_make_sub(_ids + _ids[:1], [_good] * (len(_ids) + 1)))["valid"],
    "missing id fails": not validate_submission(_make_sub(_ids[1:], [_good] * (len(_ids) - 1)))["valid"],
    "null prediction fails": not validate_submission(_make_sub(_ids, [None] + [_good] * (len(_ids) - 1)))["valid"],
    "invalid SMILES fails": not validate_submission(_make_sub(_ids, [["not_smiles("]] + [_good] * (len(_ids) - 1)))["valid"],
    ">25 candidates fails": (SUB_FORMAT["kind"] == "wide" and len(SUB_FORMAT["pred_cols"]) <= 25) or
                            not validate_submission(_make_sub(_ids, [[f"C{'C' * i}O" for i in range(26)]] + [_good] * (len(_ids) - 1)))["valid"],
    "connectivity duplicate warns": len(validate_submission(_make_sub(_ids, [["C[C@H](N)C(=O)O", "C[C@@H](N)C(=O)O"]] * len(_ids)))["warnings"]) > 0,
}
for k_, v_ in sub_tests.items():
    print(f"  [{'PASS' if v_ else 'FAIL'}] {k_}")
print("sample_submission itself:", validate_submission(sample_sub, check_smiles=True))
CHECKS["Submission validator tests pass"] = all(sub_tests.values())

## 37–38. Save permanent project artifacts with provenance

Parquet for tables, packed bits (`npz`) for fingerprints — never dense 2048-wide float matrices. Each artifact has a `.meta.json` sidecar (timestamp, input fingerprints, SEED, versions, config + hash, row count, sampling method where relevant).

In [ ]:
molecule_meta = molecule_df.merge(per_mol.drop(columns=["n_spectra"]).reset_index(), on="connectivity_key", how="left") \
                           .merge(connectivity_folds[["connectivity_key", "fold"]], on="connectivity_key", how="left") \
                           .merge(scaffold_folds[["connectivity_key", "scaffold_fold"]], on="connectivity_key", how="left")
spec_cols = [c for c in ["spectrum_id", "split", "row_pos", "molecule_id", "unit_id", "smiles", "connectivity_key", "precursor_mz", "precursor_flag",
                         "adduct", "adduct_supported", "adduct_z", "ion_mode", "ce_raw", "ce_value", "ce_min", "ce_max", "ce_unit", "ce_multi", "ce_key",
                         "instrument", "source", "ppm_provided", "exact_mass", "neutral_mass_obs", "mass_error_da", "mass_error_ppm",
                         "precursor_ppm", "n_peaks_all", "mz_decile", "fold", "scaffold_fold"]]
spectrum_metadata = pd.concat([train_meta[[c for c in spec_cols if c in train_meta]],
                               test_meta[[c for c in spec_cols if c in test_meta]]], ignore_index=True)
for c in ("adduct", "ion_mode", "ce_unit", "ce_key", "instrument", "source", "split", "precursor_flag", "ce_raw", "smiles", "connectivity_key", "molecule_id", "unit_id"):
    if c in spectrum_metadata:
        spectrum_metadata[c] = spectrum_metadata[c].astype(object).astype("string")
for c in ("fold", "scaffold_fold"):
    spectrum_metadata[c] = spectrum_metadata[c].fillna(-1).astype(np.int8)
for c in ("mass_error_ppm", "precursor_ppm", "mass_error_da", "ce_value"):
    if c in spectrum_metadata:
        spectrum_metadata[c] = spectrum_metadata[c].astype("float32")

save_artifact(molecule_meta, "molecule_metadata", extra={"description": "one row per connectivity_key"})
save_artifact(spectrum_metadata, "spectrum_metadata", extra={"description": "one row per spectrum (train+test); fold=-1 for test/invalid"})
save_artifact(connectivity_folds, "connectivity_folds", extra={"method": "balanced grouped split; strata = mass decile × dominant source", "n_folds": CONFIG["n_folds"]})
save_artifact(scaffold_df.merge(sg.rename(columns={"scaffold_group": "murcko_scaffold"})[["murcko_scaffold", "n_spectra"]], on="murcko_scaffold", how="left")
              .assign(scaffold_fold=lambda d: d.murcko_scaffold.map(sfold)), "scaffold_metadata")
save_artifact(scaffold_folds, "scaffold_folds", extra={"grouping": "Murcko scaffold; acyclic → own group"})
save_artifact(AMB.drop(columns=["mass_range", "complexity"]), "mass_ambiguity", extra={"tolerances_ppm": CONFIG["ambiguity_tols_ppm"], "library": "training molecules"})
save_array_artifact({"fp_packed": FP_PACKED, "connectivity_key": molecule_df.connectivity_key.to_numpy().astype("U14")}, "morgan_r2_2048",
                    extra={"radius": 2, "fpSize": 2048, "generator": "rdFingerprintGenerator.GetMorganGenerator", "packing": "np.packbits(axis=1)"})
feat_all = pd.concat([train_s[["spectrum_id", "split", "sample_component"] + FEAT_COLS], test_s[["spectrum_id", "split", "sample_component"] + FEAT_COLS]], ignore_index=True)
save_artifact(feat_all, "spectral_features_sample", extra={"sampling": SAMPLE_INFO})
save_artifact(PAIRS.assign(spectrum_id_i=S.spectrum_id.values[PAIRS.i], spectrum_id_j=S.spectrum_id.values[PAIRS.j]), "pair_similarity_study",
              extra={"sampling": SAMPLE_INFO, "tol_ppm": CONFIG["pair_mass_tol_ppm"]})
save_artifact(shift_tbl, "train_test_shift")
if len(src_tbl):
    save_artifact(src_tbl.reset_index(), "source_vs_test")

_required = ["molecule_metadata", "spectrum_metadata", "connectivity_folds", "scaffold_metadata", "mass_ambiguity"]
_exists = {n: (CACHE_ROOT / f"{n}.parquet").exists() and (CACHE_ROOT / f"{n}.meta.json").exists() for n in _required}
_fresh = {n: artifact_is_fresh(n) for n in _required}
_stale_detect = not artifact_is_fresh("molecule_metadata", config={**CONFIG, "n_folds": CONFIG["n_folds"] + 1})
print("exists:", _exists); print("fresh:", _fresh); print("stale cache detected when config changes:", _stale_detect)
check("Artifacts saved", all(_exists.values()) and all(_fresh.values()) and _stale_detect)
display(pd.DataFrame([{"artifact": k, "path": v, "MB": round(Path(v).stat().st_size / 1e6, 2)} for k, v in ARTIFACTS.items()]))  # noqa: F821

## 39. Final modelling decision table

Each row is emitted **only if the measured evidence supports it**; the evidence column quotes the numbers.

In [ ]:
def _r(sec, key, default=np.nan):
    return RESULTS.get(sec, {}).get(key, default)


decisions = []
med_amb = float(AMB[rt].median())
decisions.append(("Accurate mass narrows the search space", f"median {med_amb:.0f} / p90 {AMB[rt].quantile(.9):.0f} other library structures within ±{RECOMMENDED_PPM:g} ppm (library of {N_MOL:,})",
                  "Candidate generation starts from consensus neutral mass with the measured tolerance."))
decisions.append(("Precursor mass accuracy is measured, not assumed", f"±{RECOMMENDED_PPM:g} ppm captures {_r('§15 mass accuracy', 'capture at recommended'):.1%} of training spectra",
                  f"Use ±{RECOMMENDED_PPM:g} ppm (neutral space) as the default window; report recall at neighbouring tolerances."))
if (per_mol_formula > 1).mean() > 0.2:
    decisions.append(("Many structures share formulas", f"{(per_mol_formula > 1).mean():.0%} of molecules share their formula; p90 sharing = {per_mol_formula.quantile(.9):.0f}",
                      "Formula prediction alone is insufficient; the ranker must discriminate isomers from fragments."))
if len(SEP_TBL):
    a_easy = SEP_TBL.AUC.iloc[0]; a_hard = SEP_TBL.AUC.iloc[-1]
    if a_hard >= 0.7:
        decisions.append(("Spectra carry structural signal beyond mass/acquisition", f"AUC {a_easy:.2f} (random) → {a_hard:.2f} ({SEP_TBL['negative set'].iloc[-1]})",
                          "Spectral retrieval / learned spectrum–structure matching is worth building."))
    if a_easy - a_hard > 0.1:
        decisions.append(("Easy negatives overstate spectral separability", f"AUC drops by {a_easy - a_hard:.2f} from random to mass-matched negatives",
                          "Evaluate and mine hard negatives among mass-matched candidates only."))
if ADDUCT_EFFECT.get("same adduct") and ADDUCT_EFFECT.get("different adduct"):
    d_med = ADDUCT_EFFECT["same adduct"]["median_cos"] - ADDUCT_EFFECT["different adduct"]["median_cos"]
    if d_med > 0.1:
        decisions.append(("Spectral similarity drops across adducts", f"median same-molecule cosine {ADDUCT_EFFECT['same adduct']['median_cos']:.2f} (same adduct) vs {ADDUCT_EFFECT['different adduct']['median_cos']:.2f} (different)",
                          "Adduct must be an explicit model input; cross-adduct spectral matching needs conditioning."))
sing_frac = float((fam == 1).sum() / max(1, len(fam)))
if sing_frac > 0.3 or float(np.median(nn_scaffold)) < float(np.median(nn_primary)) - 0.05:
    decisions.append(("Large long tail of structural families", f"{sing_frac:.0%} of scaffolds are singletons; val→train NN Tanimoto {np.median(nn_primary):.2f} (primary) vs {np.median(nn_scaffold):.2f} (scaffold CV)",
                      "Representation learning that generalises across scaffolds matters; track scaffold-CV alongside primary CV."))
if len(TEST_COV) and (TEST_COV > 0).mean() < 0.95:
    decisions.append(("Training library does not cover test masses", f"only {(TEST_COV > 0).mean():.0%} of test units have any training structure within tolerance",
                      "An external structure library (or generator) is mandatory for candidates."))
if large:
    decisions.append(("Train/test shift exists", f"moderate/large shift in: {', '.join(large)}",
                      "Condition models on shifted variables; add the test-like diagnostic split if available."))
decisions.append(("Repeated spectra per molecule", f"spectra/molecule median {spm.median():.0f}, p99 {spm.quantile(.99):.0f}; singletons {(spm == 1).mean():.0%}",
                  "Group all splits by connectivity; report per-molecule metrics; aggregate multi-spectrum test units into one ranked list."))
DECISION_TABLE = pd.DataFrame(decisions, columns=["EDA finding", "Evidence", "Modeling consequence"])
display(DECISION_TABLE)  # noqa: F821
save_artifact(DECISION_TABLE, "decision_table")

## 40. Specification for `02_candidate_generation.ipynb`

Every number below is read from this notebook's measurements. Notebook 02 must load the saved artifacts, not recompute them.

In [ ]:
unsup_test = int(test_meta.get("adduct_supported", pd.Series(False, index=test_meta.index)).eq(False).sum())
NB02_SPEC = {
    "inputs": {"molecule_metadata": ARTIFACTS["molecule_metadata"], "spectrum_metadata": ARTIFACTS["spectrum_metadata"],
               "connectivity_folds": ARTIFACTS["connectivity_folds"], "mass_ambiguity": ARTIFACTS["mass_ambiguity"]},
    "recommended_ppm_tolerance": RECOMMENDED_PPM,
    "tolerance_sweep_to_report": CONFIG["tolerance_grid_ppm"],
    "neutral_mass_conversion": "neutral_mass_from_precursor(precursor_mz, adduct) with explicit electron mass; "
                               "per prediction unit use the median over spectra with supported adducts; flag units with spread > 2×tolerance",
    "adduct_handling": "parse_adduct() from Notebook 01 (notation-based, electron-aware); adduct is carried as an explicit feature",
    "fallback_for_unsupported_or_missing_adducts": "enumerate the top training adducts of the same ion mode (covering ≥99% of that mode's spectra), "
                                                   "take the union of their mass windows, and tag candidates with the hypothesised adduct",
    "n_test_spectra_needing_fallback": unsup_test,
    "candidate_library_key": "connectivity_key = InChIKey first block after RDKit tautomer canonicalisation",
    "candidate_deduplication": "deduplicate by connectivity_key before ranking; keep one representative SMILES per key",
    "candidate_source": "external structure library required (training library only covers "
                        + (f"{(TEST_COV > 0).mean():.0%}" if len(TEST_COV) else "n/a") + " of test units by mass and never contains held-out structures)",
    "expected_candidate_count_train_library": {"median": med_amb, "p90": float(AMB[rt].quantile(.9)), "p99": float(AMB[rt].quantile(.99))},
    "expected_recall_target": f"candidate recall ≥ {CONFIG['target_capture']:.0%} × (library coverage) under connectivity-grouped CV; "
                              f"mass-window loss alone is bounded by 1 − {_r('§15 mass accuracy', 'capture at recommended'):.3f}",
    "evaluation": "evaluate_ranking(); denominator = ALL queries; report candidate_recall, Hit@k and MRR@25 per fold (primary) and on scaffold folds",
    "forbidden": ["re-splitting data", "dropping queries with missing truth", "raw-SMILES equality", "choosing tolerance on the leaderboard"],
}
(OUTPUT_ROOT / "notebook02_spec.json").write_text(json.dumps(NB02_SPEC, indent=2, default=str))
print(json.dumps(NB02_SPEC, indent=2, default=str))

## 41. Executive summary (generated from measured results)

In [ ]:
def _fmt(v):
    return f"{v:.3g}" if isinstance(v, float) else str(v)


summary = f"""
DATASET STRUCTURE
  One row = one MS/MS spectrum. {len(train_meta):,} train spectra → {N_MOL:,} connectivities ({spm.median():.0f} median, {spm.max()} max spectra/molecule; {(spm == 1).mean():.0%} singletons).
  Test: {len(test_meta):,} spectra in {test_meta.unit_id.nunique():,} prediction units ('{SUB_ID_COL}').

CHEMISTRY REPRESENTATION
  Identity = InChIKey first block after RDKit tautomer canonicalisation ({sum(CONN_TESTS.values())}/{len(CONN_TESTS)} identity tests pass).
  {len(struct_df):,} raw SMILES → {N_MOL:,} connectivities; {n_invalid} invalid SMILES flagged.

SPECTRAL CHARACTERISTICS
  Median peaks: train {train_s.n_peaks.median():.0f}, test {test_s.n_peaks.median():.0f}; Spearman(mass, peaks) = {rho_tr:.2f}.
  Same-molecule vs mass-matched different-molecule AUC: {', '.join(f'{k}: {v:.2f}' for k, v in auc.items())}.
  Best baseline similarity: {best_sim.method} with '{bt}' intensities.

MASS ACCURACY
  ±{RECOMMENDED_PPM:g} ppm (neutral) captures {_fmt(_r('§15 mass accuracy', 'capture at recommended'))} of training spectra. Physics check: {'PASS' if PHYSICS_AGREES else 'FAIL'}.

CANDIDATE AMBIGUITY
  Within ±{RECOMMENDED_PPM:g} ppm: median {med_amb:.0f}, p90 {AMB[rt].quantile(.9):.0f} other training structures. Test units with any training structure in window: {(f'{(TEST_COV > 0).mean():.0%}' if len(TEST_COV) else 'n/a')}.

STRUCTURAL DIVERSITY
  {len(fam):,} Murcko scaffolds ({(fam == 1).mean():.0%} singletons), {acyclic.mean():.0%} acyclic; NN Tanimoto median {nn_df.nn1_tanimoto.median():.2f}.

TRAIN/TEST SHIFT
  Moderate/large: {', '.join(large) if large else 'none'}. Most test-like source: {TESTLIKE_SOURCE or 'no clear winner'}.

VALIDATION
  Permanent 5-fold connectivity-grouped CV (leakage 0) + scaffold-grouped hard CV{' + test-like holdout' if TESTLIKE_SOURCE else ''}.

MODELING IMPLICATIONS
""" + "\n".join(f"  • {r['EDA finding']} → {r['Modeling consequence']}" for _, r in DECISION_TABLE.iterrows())
print(summary)
(OUTPUT_ROOT / "executive_summary.txt").write_text(summary)

## 42. Required clean-run check

"Clean-kernel run succeeds" is verified mechanically: the first code cell must have run at `execution_count == 1` (fresh kernel), and the number of executions since then must equal the number of code cells in the notebook (every cell ran exactly once, in order). If you ran cells interactively, use **Restart Kernel → Run All** and save with outputs.

In [ ]:
N_CODE_CELLS = 45
if _IP is not None:
    _executed = len(_IP.user_ns["In"]) - _HIST_AT_START + 1
    clean = _FRESH_KERNEL and (_executed == N_CODE_CELLS)
else:
    _executed, clean = None, False
_fig_ok = len(FIGURES) > 0 and all(Path(f).exists() for f in FIGURES)
CHECKS["Key figures exist"] = _fig_ok
CHECKS["Clean-kernel run succeeds"] = clean and all(v for k, v in CHECKS.items() if k != "Clean-kernel run succeeds")
ORDER = ["Dataset mapped", "Competition connectivity validated", "Adduct algebra validated", "Adduct physics matches data",
         "Mass-error analysis complete", "Spectral quality audited", "Candidate ambiguity measured", "Train/test shift quantified",
         "Connectivity CV built", "CV leakage = 0", "Ranking metric tests pass", "Submission validator tests pass",
         "Artifacts saved", "Key figures exist", "Clean-kernel run succeeds"]
CHECKS["Scaffold analysis complete"] = "§26 scaffolds" in RESULTS and "§33 scaffold CV" in RESULTS
ORDER.insert(6, "Scaffold analysis complete")
final = pd.DataFrame({"requirement": ORDER, "status": ["PASS" if CHECKS.get(k) else "FAIL" for k in ORDER]})
print(final.to_string(index=False))
print(f"\nfresh kernel: {_FRESH_KERNEL} | cells executed: {_executed} / {N_CODE_CELLS} | figures: {len(FIGURES)}")
(OUTPUT_ROOT / "nb01_checklist.json").write_text(json.dumps({k: bool(CHECKS.get(k)) for k in ORDER}, indent=2))
NOTEBOOK_01_COMPLETE = bool((final.status == "PASS").all())
print("\nNOTEBOOK 01 COMPLETE:", NOTEBOOK_01_COMPLETE)